# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/space-0d/flyrank/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found")

print("HF token loaded successfully.")

HF token loaded successfully.


In [9]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [10]:
# Inspect the full warehouse schema

for name, src in TABLES.items():
    print(f"\n{'='*60}")
    print(name)
    print('='*60)

    con.sql(f"DESCRIBE SELECT * FROM {src}").show()


dim_clients
┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_active           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_gsc_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_ga4_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ access_profile      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_created_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_updated_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_start      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_start      │ DATE        │ YES     │ NUL

In [11]:
# Show a few rows from each table

for name, src in TABLES.items():
    print(f"\n{'='*60}")
    print(name)
    print('='*60)

    con.sql(f"SELECT * FROM {src} LIMIT 3").show()


dim_clients
┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false          │ false     

In [14]:
feature_sql = f"""
WITH max_date AS (
    SELECT MAX(report_date) AS snapshot_date
    FROM {TABLES["fact_daily"]}
),

daily_features AS (
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        m.snapshot_date,

        -- 90-day totals
        SUM(d.gsc_impressions) AS impressions_90d,
        SUM(d.gsc_clicks) AS clicks_90d,

        -- Average ranking position
        AVG(
            CASE
                WHEN d.gsc_impressions > 0
                THEN d.gsc_avg_position
            END
        ) AS avg_position_90d,

        -- Last 30 days
        SUM(
            CASE
                WHEN d.report_date >= m.snapshot_date - INTERVAL '30 days'
                THEN d.gsc_impressions
                ELSE 0
            END
        ) AS impressions_last30,

        SUM(
            CASE
                WHEN d.report_date >= m.snapshot_date - INTERVAL '30 days'
                THEN d.gsc_clicks
                ELSE 0
            END
        ) AS clicks_last30

    FROM {TABLES["fact_daily"]} AS d

    CROSS JOIN max_date AS m

    WHERE d.gsc_data_available = TRUE

    GROUP BY
        d.client_hash_id,
        d.content_hash_id,
        m.snapshot_date
)

SELECT
    d.*,

    c.content_created_date,
    c.content_updated_date,
    c.last_optimized_date,
    c.content_type,
    c.search_volume,
    c.competition,
    c.competition_level,
    c.main_intent,
    c.backlinks,
    c.category_count,
    c.char_count,
    c.word_count,
    c.is_published,
    c.is_deleted,

    -- 90-day CTR
    CASE
        WHEN d.impressions_90d > 0
        THEN d.clicks_90d * 1.0 / d.impressions_90d
        ELSE 0
    END AS ctr_90d,

    -- Last-30-day CTR
    CASE
        WHEN d.impressions_last30 > 0
        THEN d.clicks_last30 * 1.0 / d.impressions_last30
        ELSE 0
    END AS ctr_last30,

    -- Content age
    DATE_DIFF(
        'day',
        c.content_created_date,
        d.snapshot_date
    ) AS content_age_days,

    -- Days since content update
    DATE_DIFF(
        'day',
        c.content_updated_date,
        d.snapshot_date
    ) AS days_since_update

FROM daily_features AS d

INNER JOIN {TABLES["dim_content"]} AS c
    ON d.client_hash_id = c.client_hash_id
    AND d.content_hash_id = c.content_hash_id

WHERE
    c.is_deleted = FALSE
    AND c.is_published = TRUE
"""

features = con.sql(feature_sql).df()


print("Rows:", len(features))
print("Columns:", len(features.columns))

display(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 292949
Columns: 26


,client_hash_id,content_hash_id,snapshot_date,impressions_90d,clicks_90d,avg_position_90d,impressions_last30,clicks_last30,content_created_date,content_updated_date,...,backlinks,category_count,char_count,word_count,is_published,is_deleted,ctr_90d,ctr_last30,content_age_days,days_since_update
0,client_2094c6eb080311d5,content_149355c8dfc3f8e1,2026-06-30,12.0,0.0,50.527778,0.0,0.0,2026-01-22,2026-05-12,...,0,0,28921,4038,True,False,0.000000,0.00,159,49
1,client_2094c6eb080311d5,content_1497d20f8498c13f,2026-06-30,170.0,2.0,29.015515,100.0,2.0,2026-05-05,2026-05-20,...,0,0,16953,2490,True,False,0.011765,0.02,56,41
2,client_2094c6eb080311d5,content_14a3d47ccd0d15dc,2026-06-30,10.0,0.0,6.111111,0.0,0.0,2025-12-09,2026-05-12,...,0,0,25450,3864,True,False,0.000000,0.00,203,49
3,client_2094c6eb080311d5,content_14a6f92117604fef,2026-06-30,200.0,0.0,9.190605,1.0,0.0,2025-12-11,2026-05-20,...,0,0,18112,2867,True,False,0.000000,0.00,201,41
4,client_2094c6eb080311d5,content_14a86c63a214f648,2026-06-30,160.0,0.0,23.094434,3.0,0.0,2025-12-09,2026-05-12,...,0,0,19539,2949,True,False,0.000000,0.00,203,49


In [18]:
import pandas as pd

In [20]:
print("=" * 60)
print("FEATURE TABLE QUALITY CHECK")
print("=" * 60)

print("\n1. FEATURE TABLE SIZE")
print("Rows   :", len(features))
print("Columns:", len(features.columns))


print("\n2. SNAPSHOT DATE")
print("Minimum:", features["snapshot_date"].min())
print("Maximum:", features["snapshot_date"].max())


print("\n3. MISSING VALUES")

missing = (
    features.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_pct = (
    features.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_pct.round(2)
})

display(missing_table.head(15))


print("\n4. DUPLICATE CHECK")

duplicates = features.duplicated(
    subset=["client_hash_id", "content_hash_id"]
).sum()

print("Duplicate client-content pairs:", duplicates)

print("\n5. PERFORMANCE STATISTICS")

performance_cols = [
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d",
    "impressions_last30",
    "clicks_last30",
    "ctr_90d",
    "ctr_last30",
    "content_age_days",
    "days_since_update"
]

display(
    features[performance_cols]
    .describe()
    .T
)

print("\n6. VALIDITY CHECKS")

print(
    "Negative impressions:",
    (features["impressions_90d"] < 0).sum()
)

print(
    "Negative clicks:",
    (features["clicks_90d"] < 0).sum()
)

print(
    "CTR > 1:",
    (features["ctr_90d"] > 1).sum()
)

print(
    "Published content:",
    features["is_published"].sum()
)

print(
    "Deleted content:",
    features["is_deleted"].sum()
)


print("\n" + "=" * 60)
print("=" * 60)

FEATURE TABLE QUALITY CHECK

1. FEATURE TABLE SIZE
Rows   : 292949
Columns: 26

2. SNAPSHOT DATE
Minimum: 2026-06-30 00:00:00
Maximum: 2026-06-30 00:00:00

3. MISSING VALUES


,missing_count,missing_percent
last_optimized_date,248991,84.99
backlinks,103687,35.39
char_count,62735,21.41
word_count,62735,21.41
competition_level,36820,12.57
main_intent,36140,12.34
competition,35211,12.02
search_volume,35211,12.02
client_hash_id,0,0.00
content_hash_id,0,0.00



4. DUPLICATE CHECK
Duplicate client-content pairs: 0

5. PERFORMANCE STATISTICS


,count,mean,std,min,25%,50%,75%,max
impressions_90d,292949.0,6005.306777,25776.901232,1.0,22.000000,293.000000,2572.000000,2902616.0
clicks_90d,292949.0,20.618797,131.349641,0.0,0.000000,0.000000,5.000000,24747.0
avg_position_90d,292949.0,17.119676,16.332784,0.0,6.391544,11.083408,22.359234,308.0
impressions_last30,292949.0,757.417950,4416.536264,0.0,0.000000,16.000000,263.000000,618799.0
clicks_last30,292949.0,2.896170,20.861095,0.0,0.000000,0.000000,1.000000,3880.0
ctr_90d,292949.0,0.004618,0.025172,0.0,0.000000,0.000000,0.002872,1.0
ctr_last30,292949.0,0.003028,0.024236,0.0,0.000000,0.000000,0.001208,1.0
content_age_days,292949.0,233.608365,138.051419,-1.0,112.000000,242.000000,334.000000,585.0
days_since_update,292949.0,42.657207,42.187551,-6.0,18.000000,41.000000,41.000000,394.0



6. VALIDITY CHECKS
Negative impressions: 0
Negative clicks: 0
CTR > 1: 0
Published content: 292949
Deleted content: 0



In [21]:
future_sql = f"""
WITH future_daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS future_impressions,
        SUM(gsc_clicks) AS future_clicks,
        AVG(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS future_avg_position
    FROM {TABLES["fact_daily"]}
    WHERE
        report_date > DATE '2026-06-30'
        AND report_date <= DATE '2026-07-30'
        AND gsc_data_available = TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM future_daily
"""

future = con.sql(future_sql).df()

print("Future records:", len(future))
print("Columns:", future.columns.tolist())

display(future.head())

Future records: 0
Columns: ['client_hash_id', 'content_hash_id', 'future_impressions', 'future_clicks', 'future_avg_position']


,client_hash_id,content_hash_id,future_impressions,future_clicks,future_avg_position


In [22]:
date_check = con.sql(f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS distinct_dates
FROM {TABLES["fact_daily"]}
""").df()

display(date_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,distinct_dates
0,2025-01-27,2026-06-30,520


In [23]:
cutoff_date = "2026-05-31"

feature_sql = f"""
WITH daily_features AS (
    SELECT
        d.client_hash_id,
        d.content_hash_id,

        DATE '{cutoff_date}' AS snapshot_date,

        SUM(d.gsc_impressions) AS impressions_90d,
        SUM(d.gsc_clicks) AS clicks_90d,

        AVG(
            CASE
                WHEN d.gsc_impressions > 0
                THEN d.gsc_avg_position
            END
        ) AS avg_position_90d,

        SUM(
            CASE
                WHEN d.report_date > DATE '{cutoff_date}' - INTERVAL '30 days'
                THEN d.gsc_impressions
                ELSE 0
            END
        ) AS impressions_last30,

        SUM(
            CASE
                WHEN d.report_date > DATE '{cutoff_date}' - INTERVAL '30 days'
                THEN d.gsc_clicks
                ELSE 0
            END
        ) AS clicks_last30

    FROM {TABLES["fact_daily"]} d

    WHERE
        d.report_date > DATE '{cutoff_date}' - INTERVAL '90 days'
        AND d.report_date <= DATE '{cutoff_date}'
        AND d.gsc_data_available = TRUE

    GROUP BY
        d.client_hash_id,
        d.content_hash_id
)

SELECT
    d.*,

    c.content_created_date,
    c.content_updated_date,
    c.last_optimized_date,
    c.content_type,
    c.search_volume,
    c.competition,
    c.competition_level,
    c.main_intent,
    c.backlinks,
    c.category_count,
    c.char_count,
    c.word_count,
    c.is_published,
    c.is_deleted,

    CASE
        WHEN d.impressions_90d > 0
        THEN d.clicks_90d * 1.0 / d.impressions_90d
        ELSE 0
    END AS ctr_90d,

    CASE
        WHEN d.impressions_last30 > 0
        THEN d.clicks_last30 * 1.0 / d.impressions_last30
        ELSE 0
    END AS ctr_last30,

    DATE_DIFF(
        'day',
        c.content_created_date,
        d.snapshot_date
    ) AS content_age_days,

    DATE_DIFF(
        'day',
        c.content_updated_date,
        d.snapshot_date
    ) AS days_since_update

FROM daily_features d

INNER JOIN {TABLES["dim_content"]} c
    ON d.client_hash_id = c.client_hash_id
    AND d.content_hash_id = c.content_hash_id

WHERE
    c.is_deleted = FALSE
    AND c.is_published = TRUE
"""

features = con.sql(feature_sql).df()

print("Rows:", len(features))
print("Columns:", len(features.columns))
print("Snapshot:", features["snapshot_date"].unique())

display(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 253453
Columns: 26
Snapshot: <DatetimeArray>
['2026-05-31 00:00:00']
Length: 1, dtype: datetime64[us]


,client_hash_id,content_hash_id,snapshot_date,impressions_90d,clicks_90d,avg_position_90d,impressions_last30,clicks_last30,content_created_date,content_updated_date,...,backlinks,category_count,char_count,word_count,is_published,is_deleted,ctr_90d,ctr_last30,content_age_days,days_since_update
0,client_2094c6eb080311d5,content_1497d20f8498c13f,2026-05-31,73.0,0.0,62.642882,73.0,0.0,2026-05-05,2026-05-20,...,0,0,16953,2490,True,False,0.0,0.0,26,11
1,client_2094c6eb080311d5,content_14a3d47ccd0d15dc,2026-05-31,2.0,0.0,5.500000,0.0,0.0,2025-12-09,2026-05-12,...,0,0,25450,3864,True,False,0.0,0.0,173,19
2,client_2094c6eb080311d5,content_14a6f92117604fef,2026-05-31,150.0,0.0,9.086240,15.0,0.0,2025-12-11,2026-05-20,...,0,0,18112,2867,True,False,0.0,0.0,171,11
3,client_2094c6eb080311d5,content_14a86c63a214f648,2026-05-31,86.0,0.0,26.462572,17.0,0.0,2025-12-09,2026-05-12,...,0,0,19539,2949,True,False,0.0,0.0,173,19
4,client_2094c6eb080311d5,content_14adf3543cb11d06,2026-05-31,7.0,0.0,19.500000,7.0,0.0,2026-05-15,2026-05-20,...,0,0,18205,2846,True,False,0.0,0.0,16,11


In [24]:
future_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS future_impressions,
    SUM(gsc_clicks) AS future_clicks,

    AVG(
        CASE
            WHEN gsc_impressions > 0
            THEN gsc_avg_position
        END
    ) AS future_avg_position

FROM {TABLES["fact_daily"]}

WHERE
    report_date > DATE '2026-05-31'
    AND report_date <= DATE '2026-06-30'
    AND gsc_data_available = TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

future = con.sql(future_sql).df()

print("Future records:", len(future))
print("Columns:", future.columns.tolist())

display(future.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future records: 208636
Columns: ['client_hash_id', 'content_hash_id', 'future_impressions', 'future_clicks', 'future_avg_position']


,client_hash_id,content_hash_id,future_impressions,future_clicks,future_avg_position
0,client_e547b89c05043229,content_7ee102e6f6a51610,211.0,1.0,15.552573
1,client_e547b89c05043229,content_5e1d21e744c2b861,67.0,0.0,21.256061
2,client_e547b89c05043229,content_830f2ddfc3889c38,1118.0,6.0,5.598929
3,client_e547b89c05043229,content_f43055b7a2ba7ab1,4.0,0.0,6.333333
4,client_e547b89c05043229,content_c4a360cce0f93b27,786.0,4.0,17.055149


In [25]:
import numpy as np
import pandas as pd

print("=" * 60)
print("DATASET SHAPE")
print("=" * 60)

print("Rows:", len(features))
print("Columns:", len(features.columns))
print("Snapshot:", features["snapshot_date"].unique())

print("\n" + "=" * 60)
print("DUPLICATE CLIENT + CONTENT CHECK")
print("=" * 60)

duplicates = features.duplicated(
    subset=["client_hash_id", "content_hash_id"]
).sum()

print("Duplicate client-content rows:", duplicates)

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing = (
    features.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_pct = (
    features.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_pct.round(2)
})

display(missing_report)

print("\n" + "=" * 60)
print("NUMERIC FEATURE SUMMARY")
print("=" * 60)

numeric_cols = features.select_dtypes(
    include=np.number
).columns.tolist()

display(
    features[numeric_cols]
    .describe()
    .T
)

print("\n" + "=" * 60)
print("VALUE SANITY CHECK")
print("=" * 60)

checks = {
    "negative_impressions": (
        features["impressions_90d"] < 0
    ).sum(),

    "negative_clicks": (
        features["clicks_90d"] < 0
    ).sum(),

    "negative_last30_impressions": (
        features["impressions_last30"] < 0
    ).sum(),

    "negative_last30_clicks": (
        features["clicks_last30"] < 0
    ).sum(),

    "negative_word_count": (
        features["word_count"] < 0
    ).sum(),

    "negative_backlinks": (
        features["backlinks"] < 0
    ).sum(),

    "negative_content_age": (
        features["content_age_days"] < 0
    ).sum(),

    "negative_days_since_update": (
        features["days_since_update"] < 0
    ).sum(),

    "ctr_over_1": (
        features["ctr_90d"] > 1
    ).sum(),

    "ctr_last30_over_1": (
        features["ctr_last30"] > 1
    ).sum()
}

for name, count in checks.items():
    print(f"{name}: {count}")

print("\n" + "=" * 60)
print("ZERO-IMPRESSION CONTENT")
print("=" * 60)

zero_90d = (
    features["impressions_90d"] == 0
).sum()

zero_last30 = (
    features["impressions_last30"] == 0
).sum()

print("Zero impressions (90d):", zero_90d)
print("Zero impressions (last 30d):", zero_last30)

print(
    "Zero 90d percentage:",
    round(zero_90d / len(features) * 100, 2),
    "%"
)

print(
    "Zero last30 percentage:",
    round(zero_last30 / len(features) * 100, 2),
    "%"
)

print("\n" + "=" * 60)
print("CONTENT STATUS")
print("=" * 60)

print("Published:")
print(features["is_published"].value_counts(dropna=False))

print("\nDeleted:")
print(features["is_deleted"].value_counts(dropna=False))

print("\n" + "=" * 60)
print("DATE CONSISTENCY")
print("=" * 60)

print(
    "Created after snapshot:",
    (
        features["content_created_date"]
        > features["snapshot_date"]
    ).sum()
)

print(
    "Updated after snapshot:",
    (
        features["content_updated_date"]
        > features["snapshot_date"]
    ).sum()
)

print(
    "Optimized after snapshot:",
    (
        features["last_optimized_date"]
        > features["snapshot_date"]
    ).sum()
)

print("\n" + "=" * 60)
print("KEY FEATURE QUANTILES")
print("=" * 60)

distribution_cols = [
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d",
    "impressions_last30",
    "clicks_last30",
    "ctr_90d",
    "ctr_last30",
    "search_volume",
    "backlinks",
    "word_count",
    "content_age_days",
    "days_since_update"
]

display(
    features[distribution_cols].quantile(
        [0, .01, .05, .25, .50, .75, .95, .99, 1]
    ).T
)

print("\n" + "=" * 60)
print("FUTURE DATA CHECK")
print("=" * 60)

print("Future rows:", len(future))

print(
    "Duplicate client-content:",
    future.duplicated(
        subset=["client_hash_id", "content_hash_id"]
    ).sum()
)

display(
    future[
        [
            "future_impressions",
            "future_clicks",
            "future_avg_position"
        ]
    ].describe().T
)

print("\nBasic feature checks completed.")

DATASET SHAPE
Rows: 253453
Columns: 26
Snapshot: <DatetimeArray>
['2026-05-31 00:00:00']
Length: 1, dtype: datetime64[us]

DUPLICATE CLIENT + CONTENT CHECK
Duplicate client-content rows: 0

MISSING VALUES


,missing_count,missing_percent
last_optimized_date,209618,82.70
backlinks,86795,34.25
char_count,59932,23.65
word_count,59932,23.65
competition_level,30089,11.87
main_intent,29109,11.48
competition,28490,11.24
search_volume,28490,11.24
client_hash_id,0,0.00
content_hash_id,0,0.00



NUMERIC FEATURE SUMMARY


,count,mean,std,min,25%,50%,75%,max
impressions_90d,253453.0,3245.975226,13397.476405,1.0,18.0,218.0,1549.0,2001303.0
clicks_90d,253453.0,9.99656,65.152952,0.0,0.0,0.0,3.0,17151.0
avg_position_90d,253453.0,17.126391,16.716525,0.0,6.067776,10.855203,23.211648,302.0
impressions_last30,253453.0,1016.019258,4735.335467,0.0,4.0,60.0,432.0,585502.0
clicks_last30,253453.0,3.535192,22.14579,0.0,0.0,0.0,1.0,4066.0
search_volume,224963.0,135.86692,2068.631758,0.0,0.0,10.0,20.0,368000.0
competition,224963.0,0.148917,0.281861,0.0,0.0,0.0,0.14,1.0
backlinks,166658.0,294.23787,12383.917312,0.0,0.0,0.0,0.0,2137906.0
category_count,253453.0,1.4689,2.409109,0.0,0.0,0.0,3.0,18.0
char_count,193521.0,17537.743284,7824.984117,0.0,14286.0,17755.0,20522.0,231747.0



VALUE SANITY CHECK
negative_impressions: 0
negative_clicks: 0
negative_last30_impressions: 0
negative_last30_clicks: 0
negative_word_count: 0
negative_backlinks: 0
negative_content_age: 43
negative_days_since_update: 90038
ctr_over_1: 0
ctr_last30_over_1: 0

ZERO-IMPRESSION CONTENT
Zero impressions (90d): 0
Zero impressions (last 30d): 20313
Zero 90d percentage: 0.0 %
Zero last30 percentage: 8.01 %

CONTENT STATUS
Published:
is_published
True    253453
Name: count, dtype: int64

Deleted:
is_deleted
False    253453
Name: count, dtype: int64

DATE CONSISTENCY
Created after snapshot: 43
Updated after snapshot: 90038
Optimized after snapshot: 27654

KEY FEATURE QUANTILES


,0.00,0.01,0.05,0.25,0.50,0.75,0.95,0.99,1.00
impressions_90d,1.0,1.0,1.0,18.0,218.0,1549.0,15398.0,49466.4,2001303.0
clicks_90d,0.0,0.0,0.0,0.0,0.0,3.0,44.0,165.0,17151.0
avg_position_90d,0.0,0.0,0.0,6.067776,10.855203,23.211648,53.954316,73.999722,302.0
impressions_last30,0.0,0.0,0.0,4.0,60.0,432.0,4605.0,16308.0,585502.0
clicks_last30,0.0,0.0,0.0,0.0,0.0,1.0,15.0,61.0,4066.0
ctr_90d,0.0,0.0,0.0,0.0,0.0,0.002318,0.011236,0.052632,1.0
ctr_last30,0.0,0.0,0.0,0.0,0.0,0.001622,0.011142,0.037037,1.0
search_volume,0,0,0,0,10,20,320,2400,368000
backlinks,0.0,0.0,0.0,0.0,0.0,0.0,476.0,2480.15,2137906.0
word_count,0,724,823,2218,2716,3084,4328,6249,29341



FUTURE DATA CHECK
Future rows: 208636
Duplicate client-content: 0


,count,mean,std,min,25%,50%,75%,max
future_impressions,208636.0,1036.229951,5128.574042,1.0,12.00000,91.00,511.00,615012.0
future_clicks,208636.0,5.795342,457.917380,0.0,0.00000,0.00,2.00,152170.0
future_avg_position,208636.0,22.929469,22.844836,0.0,6.93639,12.75,32.25,579.0



Basic feature checks completed.


In [26]:
cutoff_date = "2026-05-31"

feature_sql = f"""
WITH daily_features AS (
    SELECT
        d.client_hash_id,
        d.content_hash_id,

        DATE '{cutoff_date}' AS snapshot_date,

        SUM(d.gsc_impressions) AS impressions_90d,
        SUM(d.gsc_clicks) AS clicks_90d,

        AVG(
            CASE
                WHEN d.gsc_impressions > 0
                THEN d.gsc_avg_position
            END
        ) AS avg_position_90d,

        SUM(
            CASE
                WHEN d.report_date > DATE '{cutoff_date}' - INTERVAL '30 days'
                THEN d.gsc_impressions
                ELSE 0
            END
        ) AS impressions_last30,

        SUM(
            CASE
                WHEN d.report_date > DATE '{cutoff_date}' - INTERVAL '30 days'
                THEN d.gsc_clicks
                ELSE 0
            END
        ) AS clicks_last30

    FROM {TABLES["fact_daily"]} d

    WHERE
        d.report_date > DATE '{cutoff_date}' - INTERVAL '90 days'
        AND d.report_date <= DATE '{cutoff_date}'
        AND d.gsc_data_available = TRUE

    GROUP BY
        d.client_hash_id,
        d.content_hash_id
)

SELECT
    d.*,

    c.content_created_date,
    c.content_updated_date,
    c.last_optimized_date,
    c.content_type,
    c.search_volume,
    c.competition,
    c.competition_level,
    c.main_intent,
    c.backlinks,
    c.category_count,
    c.char_count,
    c.word_count,
    c.is_published,
    c.is_deleted,

    CASE
        WHEN d.impressions_90d > 0
        THEN d.clicks_90d * 1.0 / d.impressions_90d
        ELSE 0
    END AS ctr_90d,

    CASE
        WHEN d.impressions_last30 > 0
        THEN d.clicks_last30 * 1.0 / d.impressions_last30
        ELSE 0
    END AS ctr_last30,

    DATE_DIFF(
        'day',
        c.content_created_date,
        d.snapshot_date
    ) AS content_age_days,

    DATE_DIFF(
        'day',
        c.content_updated_date,
        d.snapshot_date
    ) AS days_since_update

FROM daily_features d

INNER JOIN {TABLES["dim_content"]} c
    ON d.client_hash_id = c.client_hash_id
    AND d.content_hash_id = c.content_hash_id

WHERE
    c.is_deleted = FALSE
    AND c.is_published = TRUE
    AND c.content_created_date <= DATE '{cutoff_date}'
    AND c.content_updated_date <= DATE '{cutoff_date}'
"""

features = con.sql(feature_sql).df()

print("Rows:", len(features))
print("Columns:", len(features.columns))
print("Snapshot:", features["snapshot_date"].unique())

print(
    "Negative content age:",
    (features["content_age_days"] < 0).sum()
)

print(
    "Negative days since update:",
    (features["days_since_update"] < 0).sum()
)

display(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 163415
Columns: 26
Snapshot: <DatetimeArray>
['2026-05-31 00:00:00']
Length: 1, dtype: datetime64[us]
Negative content age: 0
Negative days since update: 0


,client_hash_id,content_hash_id,snapshot_date,impressions_90d,clicks_90d,avg_position_90d,impressions_last30,clicks_last30,content_created_date,content_updated_date,...,backlinks,category_count,char_count,word_count,is_published,is_deleted,ctr_90d,ctr_last30,content_age_days,days_since_update
0,client_06d356715a8ff3b6,content_01ad5f3e74c28a0d,2026-05-31,1148.0,5.0,9.426241,1148.0,5.0,2026-05-09,2026-05-29,...,0,0,14024,2061,True,False,0.004355,0.004355,22,2
1,client_06d356715a8ff3b6,content_0426b12d88f430c7,2026-05-31,541.0,1.0,14.740305,541.0,1.0,2026-05-09,2026-05-29,...,1,2,13832,2029,True,False,0.001848,0.001848,22,2
2,client_06d356715a8ff3b6,content_056cb067dfc74b45,2026-05-31,867.0,9.0,5.934216,791.0,8.0,2026-04-21,2026-05-29,...,0,3,13607,2144,True,False,0.010381,0.010114,40,2
3,client_06d356715a8ff3b6,content_05bc1838ff9e0a23,2026-05-31,627.0,3.0,27.453440,627.0,3.0,2026-05-08,2026-05-29,...,0,0,14970,2359,True,False,0.004785,0.004785,23,2
4,client_06d356715a8ff3b6,content_07d82d3bad54da30,2026-05-31,560.0,0.0,7.353227,524.0,0.0,2026-04-24,2026-05-20,...,0,6,12184,1839,True,False,0.000000,0.000000,37,11


In [27]:
dataset = features.merge(
    future,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

dataset["has_future_data"] = dataset["future_impressions"].notna()

dataset[
    [
        "future_impressions",
        "future_clicks",
        "future_avg_position"
    ]
] = dataset[
    [
        "future_impressions",
        "future_clicks",
        "future_avg_position"
    ]
].fillna(0)

print("Dataset rows:", len(dataset))
print("Dataset columns:", len(dataset.columns))

print("\nFuture data coverage:")
print(dataset["has_future_data"].value_counts())

print("\nFuture data coverage (%):")
print(
    dataset["has_future_data"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(dataset.head())

Dataset rows: 163415
Dataset columns: 30

Future data coverage:
has_future_data
True     107602
False     55813
Name: count, dtype: int64

Future data coverage (%):
has_future_data
True     65.85
False    34.15
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,snapshot_date,impressions_90d,clicks_90d,avg_position_90d,impressions_last30,clicks_last30,content_created_date,content_updated_date,...,is_published,is_deleted,ctr_90d,ctr_last30,content_age_days,days_since_update,future_impressions,future_clicks,future_avg_position,has_future_data
0,client_06d356715a8ff3b6,content_01ad5f3e74c28a0d,2026-05-31,1148.0,5.0,9.426241,1148.0,5.0,2026-05-09,2026-05-29,...,True,False,0.004355,0.004355,22,2,797.0,2.0,10.594463,True
1,client_06d356715a8ff3b6,content_0426b12d88f430c7,2026-05-31,541.0,1.0,14.740305,541.0,1.0,2026-05-09,2026-05-29,...,True,False,0.001848,0.001848,22,2,207.0,0.0,16.506410,True
2,client_06d356715a8ff3b6,content_056cb067dfc74b45,2026-05-31,867.0,9.0,5.934216,791.0,8.0,2026-04-21,2026-05-29,...,True,False,0.010381,0.010114,40,2,1252.0,5.0,7.442776,True
3,client_06d356715a8ff3b6,content_05bc1838ff9e0a23,2026-05-31,627.0,3.0,27.453440,627.0,3.0,2026-05-08,2026-05-29,...,True,False,0.004785,0.004785,23,2,515.0,4.0,19.613708,True
4,client_06d356715a8ff3b6,content_07d82d3bad54da30,2026-05-31,560.0,0.0,7.353227,524.0,0.0,2026-04-24,2026-05-20,...,True,False,0.000000,0.000000,37,11,816.0,1.0,7.958260,True


In [28]:
import numpy as np
import pandas as pd

print("=" * 60)
print("DATASET SHAPE")
print("=" * 60)

print("Rows:", len(features))
print("Columns:", len(features.columns))
print("Snapshot:", features["snapshot_date"].unique())

print("\n" + "=" * 60)
print("DUPLICATE CLIENT + CONTENT CHECK")
print("=" * 60)

print(
    "Duplicate client-content rows:",
    features.duplicated(
        subset=["client_hash_id", "content_hash_id"]
    ).sum()
)

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing_report = pd.DataFrame({
    "missing_count": features.isna().sum(),
    "missing_percent": (
        features.isna()
        .mean()
        .mul(100)
        .round(2)
    )
}).sort_values("missing_count", ascending=False)

display(missing_report)

print("\n" + "=" * 60)
print("VALUE SANITY CHECK")
print("=" * 60)

checks = {
    "negative_impressions": (
        features["impressions_90d"] < 0
    ).sum(),

    "negative_clicks": (
        features["clicks_90d"] < 0
    ).sum(),

    "negative_last30_impressions": (
        features["impressions_last30"] < 0
    ).sum(),

    "negative_last30_clicks": (
        features["clicks_last30"] < 0
    ).sum(),

    "negative_word_count": (
        features["word_count"] < 0
    ).sum(),

    "negative_backlinks": (
        features["backlinks"] < 0
    ).sum(),

    "negative_content_age": (
        features["content_age_days"] < 0
    ).sum(),

    "negative_days_since_update": (
        features["days_since_update"] < 0
    ).sum(),

    "ctr_over_1": (
        features["ctr_90d"] > 1
    ).sum(),

    "ctr_last30_over_1": (
        features["ctr_last30"] > 1
    ).sum()
}

for name, count in checks.items():
    print(f"{name}: {count}")

print("\n" + "=" * 60)
print("ZERO-IMPRESSION CONTENT")
print("=" * 60)

zero_90d = (
    features["impressions_90d"] == 0
).sum()

zero_last30 = (
    features["impressions_last30"] == 0
).sum()

print("Zero impressions (90d):", zero_90d)
print("Zero impressions (last 30d):", zero_last30)

print(
    "Zero 90d percentage:",
    round(zero_90d / len(features) * 100, 2),
    "%"
)

print(
    "Zero last30 percentage:",
    round(zero_last30 / len(features) * 100, 2),
    "%"
)

print("\n" + "=" * 60)
print("DATE CONSISTENCY")
print("=" * 60)

print(
    "Created after snapshot:",
    (
        features["content_created_date"]
        > features["snapshot_date"]
    ).sum()
)

print(
    "Updated after snapshot:",
    (
        features["content_updated_date"]
        > features["snapshot_date"]
    ).sum()
)

print(
    "Optimized after snapshot:",
    (
        features["last_optimized_date"]
        > features["snapshot_date"]
    ).sum()
)

print("\n" + "=" * 60)
print("NUMERIC SUMMARY")
print("=" * 60)

numeric_cols = features.select_dtypes(
    include=np.number
).columns

display(
    features[numeric_cols]
    .describe()
    .T
)

print("\nBasic feature checks completed.")

DATASET SHAPE
Rows: 163415
Columns: 26
Snapshot: <DatetimeArray>
['2026-05-31 00:00:00']
Length: 1, dtype: datetime64[us]

DUPLICATE CLIENT + CONTENT CHECK
Duplicate client-content rows: 0

MISSING VALUES


,missing_count,missing_percent
last_optimized_date,158255,96.84
backlinks,64414,39.42
char_count,45795,28.02
word_count,45795,28.02
competition_level,27659,16.93
main_intent,26967,16.50
competition,26563,16.25
search_volume,26563,16.25
client_hash_id,0,0.00
content_hash_id,0,0.00



VALUE SANITY CHECK
negative_impressions: 0
negative_clicks: 0
negative_last30_impressions: 0
negative_last30_clicks: 0
negative_word_count: 0
negative_backlinks: 0
negative_content_age: 0
negative_days_since_update: 0
ctr_over_1: 0
ctr_last30_over_1: 0

ZERO-IMPRESSION CONTENT
Zero impressions (90d): 0
Zero impressions (last 30d): 14701
Zero 90d percentage: 0.0 %
Zero last30 percentage: 9.0 %

DATE CONSISTENCY
Created after snapshot: 0
Updated after snapshot: 0
Optimized after snapshot: 0

NUMERIC SUMMARY


,count,mean,std,min,25%,50%,75%,max
impressions_90d,163415.0,1559.100095,7221.309569,1.0,8.0,106.0,751.0,495880.0
clicks_90d,163415.0,4.926066,33.78726,0.0,0.0,0.0,1.0,3595.0
avg_position_90d,163415.0,17.881123,18.078521,0.0,5.727393,10.925926,24.786005,287.0
impressions_last30,163415.0,495.661249,2620.460819,0.0,2.0,30.0,230.0,179822.0
clicks_last30,163415.0,1.864229,12.522172,0.0,0.0,0.0,0.0,1175.0
search_volume,136852.0,115.136059,1348.800053,0.0,0.0,10.0,20.0,201000.0
competition,136852.0,0.144403,0.280422,0.0,0.0,0.0,0.14,1.0
backlinks,99001.0,281.501318,9339.846302,0.0,0.0,0.0,0.0,1132887.0
category_count,163415.0,1.261445,2.323983,0.0,0.0,0.0,2.0,18.0
char_count,117620.0,17297.409692,9067.174555,40.0,9777.0,17920.0,20911.0,129343.0



Basic feature checks completed.


In [29]:
dataset_labeled = dataset[
    dataset["has_future_data"]
].copy()

dataset_labeled["impression_change_pct"] = np.where(
    dataset_labeled["impressions_last30"] > 0,
    (
        (
            dataset_labeled["future_impressions"]
            - dataset_labeled["impressions_last30"]
        )
        / dataset_labeled["impressions_last30"]
    ) * 100,
    np.nan
)

dataset_labeled["click_change_pct"] = np.where(
    dataset_labeled["clicks_last30"] > 0,
    (
        (
            dataset_labeled["future_clicks"]
            - dataset_labeled["clicks_last30"]
        )
        / dataset_labeled["clicks_last30"]
    ) * 100,
    np.nan
)

print("Rows with future data:", len(dataset_labeled))

print("\nImpression change percentage:")
display(
    dataset_labeled[
        "impression_change_pct"
    ].describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)

print("\nClick change percentage:")
display(
    dataset_labeled[
        "click_change_pct"
    ].describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)

Rows with future data: 107602

Impression change percentage:


,impression_change_pct
count,105823.000000
mean,0.962703
std,593.410088
min,-99.987633
1%,-97.979798
5%,-93.243243
10%,-87.500000
25%,-70.923175
50%,-43.343109
75%,0.000000



Click change percentage:


,click_change_pct
count,36735.000000
mean,-5.147639
std,142.038262
min,-100.000000
1%,-100.000000
5%,-100.000000
10%,-100.000000
25%,-100.000000
50%,-42.857143
75%,7.692308


In [30]:
threshold_check = dataset_labeled[
    dataset_labeled["impressions_last30"] > 0
].copy()

thresholds = []

for min_impressions in [10, 20, 50, 100]:
    eligible = threshold_check[
        threshold_check["impressions_last30"] >= min_impressions
    ]

    for decline in [30, 40, 50, 60]:
        declined = eligible[
            eligible["impression_change_pct"] <= -decline
        ]

        thresholds.append({
            "min_last30_impressions": min_impressions,
            "decline_threshold": f"{decline}%",
            "eligible_pages": len(eligible),
            "declined_pages": len(declined),
            "decline_rate_percent": round(
                len(declined) / len(eligible) * 100,
                2
            )
        })

threshold_report = pd.DataFrame(thresholds)

display(threshold_report)

,min_last30_impressions,decline_threshold,eligible_pages,declined_pages,decline_rate_percent
0,10,30%,92164,56526,61.33
1,10,40%,92164,50109,54.37
2,10,50%,92164,42579,46.20
3,10,60%,92164,33933,36.82
4,20,30%,84720,51470,60.75
5,20,40%,84720,45421,53.61
6,20,50%,84720,38316,45.23
7,20,60%,84720,30211,35.66
8,50,30%,71074,42231,59.42
9,50,40%,71074,36893,51.91


In [32]:
dataset = features.merge(
    future,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

dataset["has_future_data"] = (
    dataset["future_impressions"].notna()
)

dataset[
    [
        "future_impressions",
        "future_clicks",
        "future_avg_position"
    ]
] = dataset[
    [
        "future_impressions",
        "future_clicks",
        "future_avg_position"
    ]
].fillna(0)

dataset_labeled = dataset[
    dataset["has_future_data"]
].copy()

dataset_labeled["impression_change_pct"] = np.where(
    dataset_labeled["impressions_last30"] > 0,
    (
        (
            dataset_labeled["future_impressions"]
            - dataset_labeled["impressions_last30"]
        )
        / dataset_labeled["impressions_last30"]
    ) * 100,
    np.nan
)

In [33]:
threshold_check = dataset_labeled[
    dataset_labeled["impressions_last30"] > 0
].copy()

thresholds = []

for min_impressions in [10, 20, 50, 100]:
    eligible = threshold_check[
        threshold_check["impressions_last30"] >= min_impressions
    ]

    for decline in [30, 40, 50, 60]:
        declined = eligible[
            eligible["impression_change_pct"] <= -decline
        ]

        thresholds.append({
            "min_last30_impressions": min_impressions,
            "decline_threshold": f"{decline}%",
            "eligible_pages": len(eligible),
            "declined_pages": len(declined),
            "decline_rate_percent": round(
                len(declined) / len(eligible) * 100,
                2
            )
        })

threshold_report = pd.DataFrame(thresholds)

display(threshold_report)

,min_last30_impressions,decline_threshold,eligible_pages,declined_pages,decline_rate_percent
0,10,30%,92164,56526,61.33
1,10,40%,92164,50109,54.37
2,10,50%,92164,42579,46.20
3,10,60%,92164,33933,36.82
4,20,30%,84720,51470,60.75
5,20,40%,84720,45421,53.61
6,20,50%,84720,38316,45.23
7,20,60%,84720,30211,35.66
8,50,30%,71074,42231,59.42
9,50,40%,71074,36893,51.91


In [34]:
dataset_model = dataset[
    dataset["has_future_data"] &
    (dataset["impressions_last30"] >= 50)
].copy()

dataset_model["refresh_opportunity"] = (
    dataset_model["future_impressions"]
    <= dataset_model["impressions_last30"] * 0.50
).astype(int)

print("Dataset rows:", len(dataset_model))

print("\nLabel distribution:")
print(
    dataset_model["refresh_opportunity"]
    .value_counts()
    .sort_index()
)

print("\nLabel distribution (%):")
print(
    (
        dataset_model["refresh_opportunity"]
        .value_counts(normalize=True)
        .sort_index() * 100
    ).round(2)
)

display(
    dataset_model[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_last30",
            "future_impressions",
            "refresh_opportunity"
        ]
    ].head()
)

Dataset rows: 71074

Label distribution:
refresh_opportunity
0    40476
1    30598
Name: count, dtype: int64

Label distribution (%):
refresh_opportunity
0    56.95
1    43.05
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,impressions_last30,future_impressions,refresh_opportunity
0,client_06d356715a8ff3b6,content_01ad5f3e74c28a0d,1148.0,797.0,0
1,client_06d356715a8ff3b6,content_0426b12d88f430c7,541.0,207.0,1
2,client_06d356715a8ff3b6,content_056cb067dfc74b45,791.0,1252.0,0
3,client_06d356715a8ff3b6,content_05bc1838ff9e0a23,627.0,515.0,0
4,client_06d356715a8ff3b6,content_07d82d3bad54da30,524.0,816.0,0


In [35]:
print("=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

future_columns = [
    "future_impressions",
    "future_clicks",
    "future_avg_position",
    "has_future_data",
    "refresh_opportunity",
    "impression_change_pct",
    "click_change_pct"
]

print("Future/label columns present in dataset:")
print([
    col for col in future_columns
    if col in dataset_model.columns
])

print("\nFeature columns:")
for col in dataset_model.columns:
    print(col)

LEAKAGE CHECK
Future/label columns present in dataset:
['future_impressions', 'future_clicks', 'future_avg_position', 'has_future_data', 'refresh_opportunity']

Feature columns:
client_hash_id
content_hash_id
snapshot_date
impressions_90d
clicks_90d
avg_position_90d
impressions_last30
clicks_last30
content_created_date
content_updated_date
last_optimized_date
content_type
search_volume
competition
competition_level
main_intent
backlinks
category_count
char_count
word_count
is_published
is_deleted
ctr_90d
ctr_last30
content_age_days
days_since_update
future_impressions
future_clicks
future_avg_position
has_future_data
refresh_opportunity


In [36]:
feature_columns = [
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d",
    "impressions_last30",
    "clicks_last30",
    "ctr_90d",
    "ctr_last30",
    "content_age_days",
    "days_since_update",
    "search_volume",
    "competition",
    "backlinks",
    "category_count",
    "char_count",
    "word_count",
    "content_type",
    "competition_level",
    "main_intent"
]

target_column = "refresh_opportunity"

X = dataset_model[feature_columns].copy()
y = dataset_model[target_column].copy()

print("=" * 60)
print("MODEL FEATURE CHECK")
print("=" * 60)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures used:")
print(X.columns.tolist())

print("\nTarget:")
print(target_column)

forbidden_columns = [
    "future_impressions",
    "future_clicks",
    "future_avg_position",
    "has_future_data",
    "refresh_opportunity",
    "impression_change_pct",
    "click_change_pct"
]

leakage_columns = [
    col for col in X.columns
    if col in forbidden_columns
]

print("\nForbidden columns found in X:")
print(leakage_columns)

if len(leakage_columns) == 0:
    print("\nLEAKAGE CHECK: PASSED")
else:
    print("\nLEAKAGE CHECK: FAILED")

MODEL FEATURE CHECK
X shape: (71074, 18)
y shape: (71074,)

Features used:
['impressions_90d', 'clicks_90d', 'avg_position_90d', 'impressions_last30', 'clicks_last30', 'ctr_90d', 'ctr_last30', 'content_age_days', 'days_since_update', 'search_volume', 'competition', 'backlinks', 'category_count', 'char_count', 'word_count', 'content_type', 'competition_level', 'main_intent']

Target:
refresh_opportunity

Forbidden columns found in X:
[]

LEAKAGE CHECK: PASSED


In [37]:
from sklearn.model_selection import GroupShuffleSplit

groups = dataset_model["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("=" * 60)
print("CLIENT-GROUPED VALIDATION")
print("=" * 60)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print(
    "Training clients:",
    groups_train.nunique()
)

print(
    "Testing clients:",
    groups_test.nunique()
)

overlap = set(groups_train) & set(groups_test)

print(
    "Client overlap:",
    len(overlap)
)

print("\nTraining label distribution:")
print(
    y_train.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nTesting label distribution:")
print(
    y_test.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

if len(overlap) == 0:
    print("\nGROUP LEAKAGE CHECK: PASSED")
else:
    print("\nGROUP LEAKAGE CHECK: FAILED")

CLIENT-GROUPED VALIDATION
Training rows: 52290
Testing rows: 18784
Training clients: 37
Testing clients: 10
Client overlap: 0

Training label distribution:
refresh_opportunity
0    55.93
1    44.07
Name: proportion, dtype: float64

Testing label distribution:
refresh_opportunity
0    59.78
1    40.22
Name: proportion, dtype: float64

GROUP LEAKAGE CHECK: PASSED


In [38]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
baseline_prob = baseline.predict_proba(X_test)[:, 1]

print("=" * 60)
print("BASELINE MODEL")
print("=" * 60)

print("Accuracy:",
      round(accuracy_score(y_test, baseline_pred), 4))

print("Precision:",
      round(precision_score(
          y_test,
          baseline_pred,
          zero_division=0
      ), 4))

print("Recall:",
      round(recall_score(
          y_test,
          baseline_pred,
          zero_division=0
      ), 4))

print("F1:",
      round(f1_score(
          y_test,
          baseline_pred,
          zero_division=0
      ), 4))

print("ROC-AUC:",
      round(roc_auc_score(
          y_test,
          baseline_prob
      ), 4))

BASELINE MODEL
Accuracy: 0.5978
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.5


In [39]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_features = [
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d",
    "impressions_last30",
    "clicks_last30",
    "ctr_90d",
    "ctr_last30",
    "content_age_days",
    "days_since_update",
    "search_volume",
    "competition",
    "backlinks",
    "category_count",
    "char_count",
    "word_count"
]

categorical_features = [
    "content_type",
    "competition_level",
    "main_intent"
]

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

rf_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=10,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Training Random Forest...")

rf_model.fit(X_train, y_train)

print("Training complete.")

Training Random Forest...
Training complete.


In [40]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("=" * 60)
print("RANDOM FOREST EVALUATION")
print("=" * 60)

print("Accuracy:",
      round(accuracy_score(y_test, rf_pred), 4))

print("Precision:",
      round(precision_score(
          y_test,
          rf_pred,
          zero_division=0
      ), 4))

print("Recall:",
      round(recall_score(
          y_test,
          rf_pred,
          zero_division=0
      ), 4))

print("F1:",
      round(f1_score(
          y_test,
          rf_pred,
          zero_division=0
      ), 4))

print("ROC-AUC:",
      round(roc_auc_score(
          y_test,
          rf_prob
      ), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

RANDOM FOREST EVALUATION
Accuracy: 0.5847
Precision: 0.4889
Recall: 0.7231
F1: 0.5834
ROC-AUC: 0.6522

Confusion Matrix:
[[5521 5709]
 [2092 5462]]


In [41]:
import numpy as np

def precision_at_k(y_true, scores, k):
    k = min(k, len(y_true))

    order = np.argsort(scores)[::-1][:k]

    return np.mean(
        np.asarray(y_true)[order]
    )

print("=" * 60)
print("PRECISION@K")
print("=" * 60)

for k in [100, 500, 1000, 2000, 5000]:
    p_at_k = precision_at_k(
        y_test.values,
        rf_prob,
        k
    )

    print(
        f"Precision@{k}:",
        round(p_at_k, 4)
    )

PRECISION@K
Precision@100: 0.67
Precision@500: 0.58
Precision@1000: 0.552
Precision@2000: 0.5615
Precision@5000: 0.5508


In [44]:
print([name for name in globals() if "forest" in name.lower() or "rf" in name.lower()])

['performance_cols', 'RandomForestClassifier', 'rf_model', 'rf_pred', 'rf_prob']


In [47]:
print(type(rf_model))
print(rf_model.named_steps)

<class 'sklearn.pipeline.Pipeline'>
{'preprocessor': ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['impressions_90d', 'clicks_90d',
                                  'avg_position_90d', 'impressions_last30',
                                  'clicks_last30', 'ctr_90d', 'ctr_last30',
                                  'content_age_days', 'days_since_update',
                                  'search_volume', 'competition', 'backlinks',
                                  'category_count', 'char_count',
                                  'word_count']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='missing',
                                                                strateg

In [48]:
import pandas as pd

preprocessor = rf_model.named_steps["preprocessor"]
classifier = rf_model.named_steps["classifier"]

feature_names = preprocessor.get_feature_names_out()

importance = pd.DataFrame({
    "feature": feature_names,
    "importance": classifier.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("=" * 60)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 60)

display(importance.head(20))

RANDOM FOREST FEATURE IMPORTANCE


,feature,importance
0,numeric__avg_position_90d,0.145908
1,numeric__content_age_days,0.092655
2,numeric__days_since_update,0.081441
3,numeric__ctr_last30,0.073683
4,numeric__char_count,0.072758
5,numeric__clicks_90d,0.071124
6,numeric__ctr_90d,0.070886
7,numeric__impressions_90d,0.067430
8,numeric__impressions_last30,0.065774
9,numeric__clicks_last30,0.065672


In [51]:
test_results = X_test.copy()

test_results["refresh_probability"] = rf_model.predict_proba(X_test)[:, 1]

# Recover original IDs using the test-set indices
test_results["client_hash_id"] = dataset.loc[X_test.index, "client_hash_id"].values
test_results["content_hash_id"] = dataset.loc[X_test.index, "content_hash_id"].values

test_results = test_results.sort_values(
    "refresh_probability",
    ascending=False
).reset_index(drop=True)

test_results["rank"] = range(1, len(test_results) + 1)

print("=" * 60)
print("TOP 20 REFRESH RECOMMENDATIONS")
print("=" * 60)

display(
    test_results[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "refresh_probability",
            "impressions_90d",
            "clicks_90d",
            "avg_position_90d",
            "impressions_last30",
            "clicks_last30",
            "ctr_90d",
            "ctr_last30",
            "content_age_days",
            "days_since_update",
            "search_volume",
            "backlinks",
            "word_count"
        ]
    ].head(20)
)

TOP 20 REFRESH RECOMMENDATIONS


,rank,client_hash_id,content_hash_id,refresh_probability,impressions_90d,clicks_90d,avg_position_90d,impressions_last30,clicks_last30,ctr_90d,ctr_last30,content_age_days,days_since_update,search_volume,backlinks,word_count
0,1,client_23a62021009f63c4,content_54b3b92802456c3f,0.871882,20256.0,12.0,34.630141,4379.0,3.0,0.000592,0.000685,89,11,0,0,3952
1,2,client_23a62021009f63c4,content_bada26f0333b186a,0.863596,12969.0,5.0,34.523755,5395.0,2.0,0.000386,0.000371,95,11,0,0,3902
2,3,client_23a62021009f63c4,content_c81aaca313704b4f,0.863310,25259.0,10.0,45.699680,5471.0,3.0,0.000396,0.000548,95,11,0,0,3969
3,4,client_e5c2aa26a8598242,content_b88c5b2a685b5965,0.858859,6277.0,5.0,30.804949,1840.0,1.0,0.000797,0.000543,135,11,0,0,3765
4,5,client_23a62021009f63c4,content_68d7bfebc3799aa5,0.857403,27032.0,13.0,38.219606,10531.0,8.0,0.000481,0.000760,86,11,0,0,3952
5,6,client_23a62021009f63c4,content_c7b938c9146b82f8,0.857043,24081.0,9.0,35.157465,6165.0,2.0,0.000374,0.000324,81,11,0,0,3783
6,7,client_e5c2aa26a8598242,content_c56835932e398b5a,0.855016,5885.0,0.0,34.423979,1306.0,0.0,0.000000,0.000000,135,11,0,0,4609
7,8,client_23a62021009f63c4,content_9afac8ac394aca4f,0.848867,10479.0,5.0,39.169321,3431.0,1.0,0.000477,0.000291,81,11,0,0,3783
8,9,client_23a62021009f63c4,content_4b629b107ed54970,0.847681,22941.0,9.0,33.452684,8635.0,2.0,0.000392,0.000232,75,11,0,0,3716
9,10,client_23a62021009f63c4,content_d08fdd23a06e49d0,0.847425,8740.0,3.0,35.850006,4661.0,2.0,0.000343,0.000429,75,11,0,0,3807


In [53]:
print("Dataset columns:")
print(dataset.columns.tolist())

print("\nX_test columns:")
print(X_test.columns.tolist())

Dataset columns:
['client_hash_id', 'content_hash_id', 'snapshot_date', 'impressions_90d', 'clicks_90d', 'avg_position_90d', 'impressions_last30', 'clicks_last30', 'content_created_date', 'content_updated_date', 'last_optimized_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'main_intent', 'backlinks', 'category_count', 'char_count', 'word_count', 'is_published', 'is_deleted', 'ctr_90d', 'ctr_last30', 'content_age_days', 'days_since_update', 'future_impressions', 'future_clicks', 'future_avg_position', 'has_future_data']

X_test columns:
['impressions_90d', 'clicks_90d', 'avg_position_90d', 'impressions_last30', 'clicks_last30', 'ctr_90d', 'ctr_last30', 'content_age_days', 'days_since_update', 'search_volume', 'competition', 'backlinks', 'category_count', 'char_count', 'word_count', 'content_type', 'competition_level', 'main_intent']


In [56]:
# Show dataframe variables currently available
[name for name, obj in globals().items()
 if hasattr(obj, "columns") and hasattr(obj, "shape")]

['features',
 'missing_table',
 'future',
 'date_check',
 'missing_report',
 'dataset',
 'dataset_labeled',
 'threshold_check',
 'eligible',
 'declined',
 'threshold_report',
 'dataset_model',
 'X',
 'X_train',
 'X_test',
 'importance',
 'test_results']

In [57]:
# ============================================================
# STEP 10: RANKED REFRESH RECOMMENDATIONS
# ============================================================

# Get refresh probability
test_prob = rf_model.predict_proba(X_test)[:, 1]

# Get corresponding original rows
recommendations = dataset_model.loc[X_test.index].copy()

# Add prediction probability
recommendations["refresh_probability"] = test_prob

# Rank highest probability first
recommendations = recommendations.sort_values(
    "refresh_probability",
    ascending=False
)

print("=" * 60)
print("TOP 20 REFRESH RECOMMENDATIONS")
print("=" * 60)

display(
    recommendations[
        [
            "client_hash_id",
            "content_hash_id",
            "refresh_probability",
            "impressions_90d",
            "clicks_90d",
            "avg_position_90d",
            "impressions_last30",
            "clicks_last30",
            "ctr_90d",
            "ctr_last30",
            "content_age_days",
            "days_since_update",
            "search_volume",
            "backlinks",
            "word_count"
        ]
    ].head(20)
)

TOP 20 REFRESH RECOMMENDATIONS


,client_hash_id,content_hash_id,refresh_probability,impressions_90d,clicks_90d,avg_position_90d,impressions_last30,clicks_last30,ctr_90d,ctr_last30,content_age_days,days_since_update,search_volume,backlinks,word_count
26643,client_23a62021009f63c4,content_54b3b92802456c3f,0.871882,20256.0,12.0,34.630141,4379.0,3.0,0.000592,0.000685,89,11,0,0,3952
31065,client_23a62021009f63c4,content_bada26f0333b186a,0.863596,12969.0,5.0,34.523755,5395.0,2.0,0.000386,0.000371,95,11,0,0,3902
106211,client_23a62021009f63c4,content_c81aaca313704b4f,0.863310,25259.0,10.0,45.699680,5471.0,3.0,0.000396,0.000548,95,11,0,0,3969
96615,client_e5c2aa26a8598242,content_b88c5b2a685b5965,0.858859,6277.0,5.0,30.804949,1840.0,1.0,0.000797,0.000543,135,11,0,0,3765
27482,client_23a62021009f63c4,content_68d7bfebc3799aa5,0.857403,27032.0,13.0,38.219606,10531.0,8.0,0.000481,0.000760,86,11,0,0,3952
106196,client_23a62021009f63c4,content_c7b938c9146b82f8,0.857043,24081.0,9.0,35.157465,6165.0,2.0,0.000374,0.000324,81,11,0,0,3783
96738,client_e5c2aa26a8598242,content_c56835932e398b5a,0.855016,5885.0,0.0,34.423979,1306.0,0.0,0.000000,0.000000,135,11,0,0,4609
29674,client_23a62021009f63c4,content_9afac8ac394aca4f,0.848867,10479.0,5.0,39.169321,3431.0,1.0,0.000477,0.000291,81,11,0,0,3783
26279,client_23a62021009f63c4,content_4b629b107ed54970,0.847681,22941.0,9.0,33.452684,8635.0,2.0,0.000392,0.000232,75,11,0,0,3716
106557,client_23a62021009f63c4,content_d08fdd23a06e49d0,0.847425,8740.0,3.0,35.850006,4661.0,2.0,0.000343,0.000429,75,11,0,0,3807


In [58]:
# ============================================================
# STEP 11: TOP-K RANKING VALIDATION
# ============================================================

# Keep only rows that have future data
eval_results = recommendations[
    recommendations["has_future_data"] == True
].copy()

# Calculate actual future impression change
eval_results["future_impression_change_pct"] = (
    (eval_results["future_impressions"] -
     eval_results["impressions_last30"])
    / eval_results["impressions_last30"].replace(0, pd.NA)
) * 100

# Define actual decline using the same 50% threshold
eval_results["actual_decline"] = (
    eval_results["future_impression_change_pct"] <= -50
)

print("=" * 60)
print("TOP-K RANKING VALIDATION")
print("=" * 60)

for k in [100, 500, 1000, 2000, 5000]:

    top_k = eval_results.head(k)

    precision = top_k["actual_decline"].mean()

    print(f"Precision@{k}: {precision:.4f}")

TOP-K RANKING VALIDATION
Precision@100: 0.6700
Precision@500: 0.5800
Precision@1000: 0.5520
Precision@2000: 0.5615
Precision@5000: 0.5508


In [59]:
print("=" * 60)
print("BASELINE VS RANDOM FOREST")
print("=" * 60)

# Actual decline rate in the test set
overall_decline_rate = eval_results["actual_decline"].mean()

print(f"Overall decline rate: {overall_decline_rate:.4f}")
print()

for k in [100, 500, 1000, 2000, 5000]:

    model_precision = eval_results.head(k)["actual_decline"].mean()

    lift = model_precision / overall_decline_rate

    print(f"Precision@{k}: {model_precision:.4f}")
    print(f"Baseline:     {overall_decline_rate:.4f}")
    print(f"Lift:         {lift:.2f}x")
    print("-" * 40)

BASELINE VS RANDOM FOREST
Overall decline rate: 0.4022

Precision@100: 0.6700
Baseline:     0.4022
Lift:         1.67x
----------------------------------------
Precision@500: 0.5800
Baseline:     0.4022
Lift:         1.44x
----------------------------------------
Precision@1000: 0.5520
Baseline:     0.4022
Lift:         1.37x
----------------------------------------
Precision@2000: 0.5615
Baseline:     0.4022
Lift:         1.40x
----------------------------------------
Precision@5000: 0.5508
Baseline:     0.4022
Lift:         1.37x
----------------------------------------


In [60]:
print("=" * 60)
print("CLIENT-LEVEL TOP-K PERFORMANCE")
print("=" * 60)

top_100 = eval_results.head(100)

client_performance = (
    top_100
    .groupby("client_hash_id")
    .agg(
        recommendations=("content_hash_id", "count"),
        actual_declines=("actual_decline", "sum"),
        precision=("actual_decline", "mean")
    )
    .sort_values("precision", ascending=False)
)

print(client_performance)

print()
print("=" * 60)
print("CLIENT COVERAGE")
print("=" * 60)

print("Clients represented in Top 100:",
      top_100["client_hash_id"].nunique())

print("Total clients in test set:",
      eval_results["client_hash_id"].nunique())

CLIENT-LEVEL TOP-K PERFORMANCE
                         recommendations  actual_declines  precision
client_hash_id                                                      
client_23a62021009f63c4               80               57     0.7125
client_65de48885f4ef01b               12                6     0.5000
client_e5c2aa26a8598242                8                4     0.5000

CLIENT COVERAGE
Clients represented in Top 100: 3
Total clients in test set: 10


In [61]:
print("=" * 60)
print("CLIENT-BALANCED TOP-K")
print("=" * 60)

client_results = []

for client, group in eval_results.groupby("client_hash_id"):

    group = group.sort_values(
        "refresh_probability",
        ascending=False
    )

    k = min(10, len(group))

    top_client = group.head(k)

    precision = top_client["actual_decline"].mean()

    client_results.append({
        "client_hash_id": client,
        "top_k": k,
        "actual_declines": int(top_client["actual_decline"].sum()),
        "precision": precision
    })

client_results = pd.DataFrame(client_results)

print(client_results.to_string(index=False))

print()
print("=" * 60)
print("AVERAGE CLIENT PRECISION")
print("=" * 60)

print(
    f"Mean Precision@10 per client: "
    f"{client_results['precision'].mean():.4f}"
)

print(
    f"Median Precision@10 per client: "
    f"{client_results['precision'].median():.4f}"
)

CLIENT-BALANCED TOP-K
         client_hash_id  top_k  actual_declines  precision
client_0b245132bb722950     10               10        1.0
client_23a62021009f63c4     10                8        0.8
client_65de48885f4ef01b     10                4        0.4
client_810019792c9b8efc     10                2        0.2
client_86ebc2f12c01f586     10                0        0.0
client_8ddc46da5414ffd8     10                1        0.1
client_9958f0a7ae1df715     10                4        0.4
client_c7c2962f1c9c3089     10                0        0.0
client_d211cb07b9059bab     10                2        0.2
client_e5c2aa26a8598242     10                4        0.4

AVERAGE CLIENT PRECISION
Mean Precision@10 per client: 0.3500
Median Precision@10 per client: 0.3000


In [62]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

# Test predictions
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

# Classification metrics
roc_auc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

# Global baseline
baseline = eval_results["actual_decline"].mean()

# Precision@K and lift
k_values = [100, 500, 1000, 2000, 5000]

rows = []

for k in k_values:
    p_at_k = eval_results.head(k)["actual_decline"].mean()

    rows.append({
        "Metric": f"Precision@{k}",
        "Value": p_at_k,
        "Baseline": baseline,
        "Lift": p_at_k / baseline
    })

summary = pd.DataFrame(rows)

print("=" * 60)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 60)

print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

print()
display(summary)

FINAL MODEL PERFORMANCE SUMMARY
ROC-AUC:   0.6522
Precision: 0.4889
Recall:    0.7231
F1:        0.5834



,Metric,Value,Baseline,Lift
0,Precision@100,0.6700,0.402151,1.666042
1,Precision@500,0.5800,0.402151,1.442245
2,Precision@1000,0.5520,0.402151,1.372620
3,Precision@2000,0.5615,0.402151,1.396243
4,Precision@5000,0.5508,0.402151,1.369636


In [63]:
# ============================================================
# STEP 16: ERROR ANALYSIS
# ============================================================

error_analysis = dataset_model.loc[X_test.index].copy()

error_analysis["predicted_probability"] = y_prob
error_analysis["predicted_label"] = y_pred
error_analysis["actual_label"] = y_test.values

# Prediction categories
error_analysis["prediction_type"] = "TN"

error_analysis.loc[
    (error_analysis["actual_label"] == 1) &
    (error_analysis["predicted_label"] == 1),
    "prediction_type"
] = "TP"

error_analysis.loc[
    (error_analysis["actual_label"] == 0) &
    (error_analysis["predicted_label"] == 1),
    "prediction_type"
] = "FP"

error_analysis.loc[
    (error_analysis["actual_label"] == 1) &
    (error_analysis["predicted_label"] == 0),
    "prediction_type"
] = "FN"

print("=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

print(
    error_analysis["prediction_type"]
    .value_counts()
)

print()
print("Prediction percentages:")

print(
    error_analysis["prediction_type"]
    .value_counts(normalize=True) * 100
)

ERROR ANALYSIS
prediction_type
FP    5709
TN    5521
TP    5462
FN    2092
Name: count, dtype: int64

Prediction percentages:
prediction_type
FP    30.392888
TN    29.392036
TP    29.077939
FN    11.137138
Name: proportion, dtype: float64


In [64]:
tp = error_analysis[
    error_analysis["prediction_type"] == "TP"
].copy()

fp = error_analysis[
    error_analysis["prediction_type"] == "FP"
].copy()

comparison = pd.DataFrame({
    "True_Positive_Mean": tp[
        [
            "impressions_90d",
            "clicks_90d",
            "avg_position_90d",
            "impressions_last30",
            "clicks_last30",
            "ctr_90d",
            "ctr_last30",
            "content_age_days",
            "days_since_update",
            "word_count"
        ]
    ].mean(),

    "False_Positive_Mean": fp[
        [
            "impressions_90d",
            "clicks_90d",
            "avg_position_90d",
            "impressions_last30",
            "clicks_last30",
            "ctr_90d",
            "ctr_last30",
            "content_age_days",
            "days_since_update",
            "word_count"
        ]
    ].mean()
})

print("=" * 60)
print("TRUE POSITIVE VS FALSE POSITIVE")
print("=" * 60)

display(comparison)

TRUE POSITIVE VS FALSE POSITIVE


,True_Positive_Mean,False_Positive_Mean
impressions_90d,3537.993043,3039.108425
clicks_90d,4.413402,4.549133
avg_position_90d,24.817434,25.377383
impressions_last30,1012.472171,799.872307
clicks_last30,1.503662,1.693817
ctr_90d,0.001241,0.001557
ctr_last30,0.001274,0.001991
content_age_days,149.648847,169.313715
days_since_update,54.189308,45.472237
word_count,4501.337045,4334.105232


In [65]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

print("=" * 60)
print("THRESHOLD TUNING")
print("=" * 60)

threshold_results = []

for threshold in np.arange(0.30, 0.71, 0.05):
    pred = (rf_prob >= threshold).astype(int)

    precision = precision_score(y_test, pred, zero_division=0)
    recall = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "predicted_positive_pct": round(pred.mean() * 100, 2)
    })

threshold_results_df = pd.DataFrame(threshold_results)

print(threshold_results_df.to_string(index=False))

THRESHOLD TUNING
 threshold  precision  recall     f1  predicted_positive_pct
      0.30     0.4348  0.9648 0.5994                   89.24
      0.35     0.4481  0.9391 0.6067                   84.28
      0.40     0.4591  0.8913 0.6060                   78.08
      0.45     0.4707  0.8197 0.5980                   70.03
      0.50     0.4889  0.7231 0.5834                   59.47
      0.55     0.5135  0.5984 0.5527                   46.86
      0.60     0.5375  0.4418 0.4850                   33.05
      0.65     0.5571  0.2711 0.3647                   19.57
      0.70     0.5552  0.1178 0.1944                    8.53


In [66]:
print("=" * 60)
print("THRESHOLD BUSINESS IMPACT")
print("=" * 60)

threshold_business = []

for threshold in np.arange(0.30, 0.71, 0.05):

    pred = (rf_prob >= threshold).astype(int)

    recommendations = pred.sum()
    actual_declines = ((pred == 1) & (y_test == 1)).sum()

    precision = (
        actual_declines / recommendations
        if recommendations > 0 else 0
    )

    threshold_business.append({
        "threshold": round(threshold, 2),
        "recommendations": int(recommendations),
        "actual_declines": int(actual_declines),
        "precision": round(precision, 4),
        "recommendation_pct": round(
            recommendations / len(y_test) * 100, 2
        )
    })

threshold_business_df = pd.DataFrame(threshold_business)

print(threshold_business_df.to_string(index=False))

THRESHOLD BUSINESS IMPACT
 threshold  recommendations  actual_declines  precision  recommendation_pct
      0.30            16762             7288     0.4348               89.24
      0.35            15831             7094     0.4481               84.28
      0.40            14667             6733     0.4591               78.08
      0.45            13155             6192     0.4707               70.03
      0.50            11171             5462     0.4889               59.47
      0.55             8803             4520     0.5135               46.86
      0.60             6208             3337     0.5375               33.05
      0.65             3676             2048     0.5571               19.57
      0.70             1603              890     0.5552                8.53


In [68]:
print("=" * 60)
print("CLIENT-BALANCED TOP-K: RANDOM FOREST")
print("=" * 60)

# Build a fresh result table from X_test and y_test
client_topk = X_test.copy()

# Add actual target
client_topk["refresh_opportunity"] = np.asarray(y_test)

# Add RF probability
client_topk["refresh_probability"] = rf_prob

# Add client ID
# X_test does not contain client_hash_id, so recover it from dataset_model
client_ids_test = dataset_model.loc[X_test.index, "client_hash_id"]

client_topk["client_hash_id"] = client_ids_test.values

# Sort by model probability within each client
client_topk = client_topk.sort_values(
    ["client_hash_id", "refresh_probability"],
    ascending=[True, False]
)

# Top 10 recommendations for each client
client_top10 = (
    client_topk
    .groupby("client_hash_id", group_keys=False)
    .head(10)
)

# Calculate client-level precision
client_balanced_results = (
    client_top10
    .groupby("client_hash_id")
    .agg(
        recommendations=("refresh_opportunity", "size"),
        actual_declines=("refresh_opportunity", "sum")
    )
)

client_balanced_results["precision"] = (
    client_balanced_results["actual_declines"] /
    client_balanced_results["recommendations"]
)

print(client_balanced_results)

print("\n" + "=" * 60)
print("AVERAGE CLIENT PRECISION")
print("=" * 60)

print(
    "Mean Precision@10 per client:",
    round(client_balanced_results["precision"].mean(), 4)
)

print(
    "Median Precision@10 per client:",
    round(client_balanced_results["precision"].median(), 4)
)

CLIENT-BALANCED TOP-K: RANDOM FOREST
                         recommendations  actual_declines  precision
client_hash_id                                                      
client_0b245132bb722950               10               10        1.0
client_23a62021009f63c4               10                8        0.8
client_65de48885f4ef01b               10                4        0.4
client_810019792c9b8efc               10                2        0.2
client_86ebc2f12c01f586               10                0        0.0
client_8ddc46da5414ffd8               10                1        0.1
client_9958f0a7ae1df715               10                4        0.4
client_c7c2962f1c9c3089               10                0        0.0
client_d211cb07b9059bab               10                2        0.2
client_e5c2aa26a8598242               10                4        0.4

AVERAGE CLIENT PRECISION
Mean Precision@10 per client: 0.35
Median Precision@10 per client: 0.3


In [69]:
print("=" * 60)
print("CLIENT-LEVEL BASELINE VS MODEL")
print("=" * 60)

client_analysis = (
    client_topk
    .groupby("client_hash_id")
    .agg(
        total_pages=("refresh_opportunity", "size"),
        actual_declines=("refresh_opportunity", "sum")
    )
)

client_analysis["baseline_decline_rate"] = (
    client_analysis["actual_declines"] /
    client_analysis["total_pages"]
)

# Merge model Precision@10
client_analysis = client_analysis.join(
    client_balanced_results[["precision"]]
    .rename(columns={"precision": "model_precision_at_10"})
)

client_analysis["lift_at_10"] = (
    client_analysis["model_precision_at_10"] /
    client_analysis["baseline_decline_rate"]
)

print(client_analysis.round(4))

CLIENT-LEVEL BASELINE VS MODEL
                         total_pages  actual_declines  baseline_decline_rate  \
client_hash_id                                                                 
client_0b245132bb722950          357              272                 0.7619   
client_23a62021009f63c4         9634             5045                 0.5237   
client_65de48885f4ef01b          744              308                 0.4140   
client_810019792c9b8efc          702              239                 0.3405   
client_86ebc2f12c01f586          915                8                 0.0087   
client_8ddc46da5414ffd8         2344              483                 0.2061   
client_9958f0a7ae1df715         1576              187                 0.1187   
client_c7c2962f1c9c3089           26                2                 0.0769   
client_d211cb07b9059bab           29                3                 0.1034   
client_e5c2aa26a8598242         2457             1007                 0.4098   

        

In [70]:
print("=" * 60)
print("CLIENT-LEVEL DATA DISTRIBUTION")
print("=" * 60)

client_distribution = (
    client_topk
    .groupby("client_hash_id")
    .agg(
        pages=("refresh_opportunity", "size"),
        declines=("refresh_opportunity", "sum"),
        decline_rate=("refresh_opportunity", "mean"),
        avg_impressions=("impressions_90d", "mean"),
        avg_position=("avg_position_90d", "mean"),
        avg_age=("content_age_days", "mean"),
        avg_days_since_update=("days_since_update", "mean"),
        avg_ctr=("ctr_90d", "mean")
    )
)

print(client_distribution.round(4).to_string())

CLIENT-LEVEL DATA DISTRIBUTION
                         pages  declines  decline_rate  avg_impressions  avg_position   avg_age  avg_days_since_update  avg_ctr
client_hash_id                                                                                                                 
client_0b245132bb722950    357       272        0.7619         611.8683       14.4997   39.6583                11.0000   0.0023
client_23a62021009f63c4   9634      5045        0.5237        4093.7089       24.0653  157.3153                62.7524   0.0022
client_65de48885f4ef01b    744       308        0.4140        1893.0013       14.8652  333.4892                59.9059   0.0036
client_810019792c9b8efc    702       239        0.3405         603.1852       12.4039   32.8832                11.0000   0.0075
client_86ebc2f12c01f586    915         8        0.0087        1958.8175       12.4428   41.8546                11.0000   0.0041
client_8ddc46da5414ffd8   2344       483        0.2061        2093.8208  

In [72]:
print("=" * 60)
print("TEST RESULTS COLUMNS")
print("=" * 60)

print(test_results.columns.tolist())
print()
print(test_results.head())

TEST RESULTS COLUMNS
['impressions_90d', 'clicks_90d', 'avg_position_90d', 'impressions_last30', 'clicks_last30', 'ctr_90d', 'ctr_last30', 'content_age_days', 'days_since_update', 'search_volume', 'competition', 'backlinks', 'category_count', 'char_count', 'word_count', 'content_type', 'competition_level', 'main_intent', 'refresh_probability', 'client_hash_id', 'content_hash_id', 'rank']

   impressions_90d  clicks_90d  avg_position_90d  impressions_last30  \
0          20256.0        12.0         34.630141              4379.0   
1          12969.0         5.0         34.523755              5395.0   
2          25259.0        10.0         45.699680              5471.0   
3           6277.0         5.0         30.804949              1840.0   
4          27032.0        13.0         38.219606             10531.0   

   clicks_last30   ctr_90d  ctr_last30  content_age_days  days_since_update  \
0            3.0  0.000592    0.000685                89                 11   
1            2.0 

In [73]:
print("=" * 60)
print("CHECK TEST LABELS")
print("=" * 60)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\ny_test distribution:")
print(pd.Series(y_test).value_counts())

print("\nFirst 10 y_test values:")
print(y_test[:10])

CHECK TEST LABELS
X_test shape: (18784, 18)
y_test shape: (18784,)

y_test distribution:
refresh_opportunity
0    11230
1     7554
Name: count, dtype: int64

First 10 y_test values:
16443    0
16450    1
16453    0
16457    0
16461    0
16467    0
16504    0
16512    0
16531    0
16535    0
Name: refresh_opportunity, dtype: int64


In [74]:
print("=" * 60)
print("ATTACHING ACTUAL LABELS")
print("=" * 60)

# Get the original test indices
test_indices = X_test.index

# Create label series indexed by the original test rows
test_labels = y_test.loc[test_indices].rename("refresh_opportunity")

# Check alignment
print("Test indices:", len(test_indices))
print("Test labels:", len(test_labels))
print("Label index matches X_test:",
      test_labels.index.equals(X_test.index))

# Add labels to test_results
test_results_calibration = test_results.copy()

# test_results was created from X_test, so align using the
# original test row ordering
test_results_calibration["refresh_opportunity"] = (
    test_labels.values
)

print("\nCalibration dataset shape:",
      test_results_calibration.shape)

print("\nColumns:")
print(test_results_calibration.columns.tolist())

print("\nLabel distribution:")
print(
    test_results_calibration["refresh_opportunity"]
    .value_counts()
)

ATTACHING ACTUAL LABELS
Test indices: 18784
Test labels: 18784
Label index matches X_test: True

Calibration dataset shape: (18784, 23)

Columns:
['impressions_90d', 'clicks_90d', 'avg_position_90d', 'impressions_last30', 'clicks_last30', 'ctr_90d', 'ctr_last30', 'content_age_days', 'days_since_update', 'search_volume', 'competition', 'backlinks', 'category_count', 'char_count', 'word_count', 'content_type', 'competition_level', 'main_intent', 'refresh_probability', 'client_hash_id', 'content_hash_id', 'rank', 'refresh_opportunity']

Label distribution:
refresh_opportunity
0    11230
1     7554
Name: count, dtype: int64


In [75]:
print("=" * 60)
print("PROBABILITY CALIBRATION CHECK")
print("=" * 60)

calibration_df = test_results_calibration[
    ["refresh_probability", "refresh_opportunity"]
].copy()

# Create probability bins
calibration_df["probability_bin"] = pd.cut(
    calibration_df["refresh_probability"],
    bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5,
          0.6, 0.7, 0.8, 0.9, 1.0],
    include_lowest=True
)

calibration_report = (
    calibration_df
    .groupby("probability_bin", observed=True)
    .agg(
        samples=("refresh_opportunity", "size"),
        mean_predicted_probability=("refresh_probability", "mean"),
        actual_decline_rate=("refresh_opportunity", "mean")
    )
    .reset_index()
)

calibration_report["calibration_gap"] = (
    calibration_report["actual_decline_rate"]
    - calibration_report["mean_predicted_probability"]
)

print(calibration_report.to_string(index=False))

PROBABILITY CALIBRATION CHECK
probability_bin  samples  mean_predicted_probability  actual_decline_rate  calibration_gap
  (-0.001, 0.1]       12                    0.083184             0.500000         0.416816
     (0.1, 0.2]      606                    0.161906             0.336634         0.174727
     (0.2, 0.3]     1404                    0.253627             0.481481         0.227854
     (0.3, 0.4]     2095                    0.353802             0.471122         0.117320
     (0.4, 0.5]     3496                    0.455266             0.394451        -0.060816
     (0.5, 0.6]     4963                    0.551667             0.219424        -0.332243
     (0.6, 0.7]     4605                    0.646471             0.515527        -0.130944
     (0.7, 0.8]     1472                    0.735021             0.537364        -0.197657
     (0.8, 0.9]      131                    0.820370             0.366412        -0.453958


In [76]:
from sklearn.metrics import brier_score_loss, log_loss

print("=" * 60)
print("PROBABILITY QUALITY")
print("=" * 60)

y_cal = test_results_calibration["refresh_opportunity"]
p_cal = test_results_calibration["refresh_probability"]

brier = brier_score_loss(y_cal, p_cal)
logloss = log_loss(y_cal, p_cal)

print(f"Brier Score: {brier:.4f}")
print(f"Log Loss:    {logloss:.4f}")

PROBABILITY QUALITY
Brier Score: 0.2725
Log Loss:    0.7431


In [77]:
from sklearn.calibration import CalibratedClassifierCV

print("=" * 60)
print("CALIBRATED RANDOM FOREST")
print("=" * 60)

calibrated_rf = CalibratedClassifierCV(
    estimator=rf_model,
    method="sigmoid",
    cv=3
)

print("Training calibrated RF...")

calibrated_rf.fit(X_train, y_train)

print("Calibrated RF training complete.")

CALIBRATED RANDOM FOREST
Training calibrated RF...
Calibrated RF training complete.


In [78]:
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss
)

print("=" * 60)
print("CALIBRATED RF EVALUATION")
print("=" * 60)

# Get calibrated probabilities
calibrated_prob = calibrated_rf.predict_proba(X_test)[:, 1]

# Actual labels
y_actual = y_test.values

# Metrics
calibrated_auc = roc_auc_score(
    y_actual,
    calibrated_prob
)

calibrated_brier = brier_score_loss(
    y_actual,
    calibrated_prob
)

calibrated_logloss = log_loss(
    y_actual,
    calibrated_prob
)

print(f"ROC-AUC:    {calibrated_auc:.4f}")
print(f"Brier Score:{calibrated_brier:.4f}")
print(f"Log Loss:   {calibrated_logloss:.4f}")

print("\nORIGINAL RF")
print(f"ROC-AUC:    0.6522")
print(f"Brier Score:0.2725")
print(f"Log Loss:   0.7431")

CALIBRATED RF EVALUATION
ROC-AUC:    0.6568
Brier Score:0.2280
Log Loss:   0.6474

ORIGINAL RF
ROC-AUC:    0.6522
Brier Score:0.2725
Log Loss:   0.7431


In [79]:
import numpy as np

print("=" * 60)
print("CALIBRATED RF PRECISION@K")
print("=" * 60)

# Sort test samples by calibrated probability
calibrated_order = np.argsort(-calibrated_prob)

y_sorted_calibrated = y_actual[calibrated_order]

k_values = [100, 500, 1000, 2000, 5000]

calibrated_precision_at_k = {}

for k in k_values:
    precision_k = y_sorted_calibrated[:k].mean()
    calibrated_precision_at_k[k] = precision_k

    print(f"Precision@{k}: {precision_k:.4f}")

CALIBRATED RF PRECISION@K
Precision@100: 0.5900
Precision@500: 0.5560
Precision@1000: 0.5800
Precision@2000: 0.5775
Precision@5000: 0.5538


In [80]:
print("=" * 60)
print("CALIBRATED RF PROBABILITY CHECK")
print("=" * 60)

calibrated_check = pd.DataFrame({
    "probability": calibrated_prob,
    "actual": y_actual
})

calibrated_check["probability_bin"] = pd.cut(
    calibrated_check["probability"],
    bins=[
        0, 0.1, 0.2, 0.3, 0.4,
        0.5, 0.6, 0.7, 0.8, 0.9, 1.0
    ],
    include_lowest=True
)

calibrated_report = (
    calibrated_check
    .groupby("probability_bin", observed=True)
    .agg(
        samples=("actual", "size"),
        mean_predicted_probability=("probability", "mean"),
        actual_decline_rate=("actual", "mean")
    )
    .reset_index()
)

calibrated_report["calibration_gap"] = (
    calibrated_report["actual_decline_rate"]
    - calibrated_report["mean_predicted_probability"]
)

print(calibrated_report.to_string(index=False))

CALIBRATED RF PROBABILITY CHECK
probability_bin  samples  mean_predicted_probability  actual_decline_rate  calibration_gap
     (0.2, 0.3]     1507                    0.268100             0.139350        -0.128750
     (0.3, 0.4]     3443                    0.353157             0.257915        -0.095243
     (0.4, 0.5]     5791                    0.452891             0.380245        -0.072646
     (0.5, 0.6]     6454                    0.548252             0.515804        -0.032447
     (0.6, 0.7]     1589                    0.621318             0.582127        -0.039191


In [81]:
print("=" * 60)
print("CALIBRATED RF THRESHOLD BUSINESS IMPACT")
print("=" * 60)

thresholds = np.arange(0.30, 0.71, 0.05)

threshold_results_calibrated = []

for threshold in thresholds:

    predicted_positive = calibrated_prob >= threshold

    recommendations = predicted_positive.sum()

    if recommendations > 0:
        actual_declines = y_actual[predicted_positive].sum()
        precision = actual_declines / recommendations
    else:
        actual_declines = 0
        precision = 0

    recommendation_pct = (
        recommendations / len(y_actual) * 100
    )

    threshold_results_calibrated.append({
        "threshold": round(threshold, 2),
        "recommendations": recommendations,
        "actual_declines": actual_declines,
        "precision": precision,
        "recommendation_pct": recommendation_pct
    })

threshold_results_calibrated = pd.DataFrame(
    threshold_results_calibrated
)

print(
    threshold_results_calibrated.to_string(index=False)
)

CALIBRATED RF THRESHOLD BUSINESS IMPACT
 threshold  recommendations  actual_declines  precision  recommendation_pct
      0.30            17277             7344   0.425074           91.977215
      0.35            15709             7018   0.446750           83.629685
      0.40            13834             6456   0.466676           73.647785
      0.45            11215             5525   0.492644           59.705068
      0.50             8043             4254   0.528907           42.818356
      0.55             4672             2599   0.556293           24.872232
      0.60             1589              925   0.582127            8.459327
      0.65              111               64   0.576577            0.590928
      0.70                0                0   0.000000            0.000000


In [83]:
print("calibrated_prob" in globals())

print([
    name for name in globals()
    if "calib" in name.lower()
])

True
['calibration_curve', 'test_results_calibration', 'calibration_df', 'calibration_report', 'CalibratedClassifierCV', 'calibrated_rf', 'calibrated_prob', 'calibrated_auc', 'calibrated_brier', 'calibrated_logloss', 'calibrated_order', 'y_sorted_calibrated', 'calibrated_precision_at_k', 'calibrated_check', 'calibrated_report', 'threshold_results_calibrated']


In [84]:
print("=" * 60)
print("ORIGINAL RF VS CALIBRATED RF - FINAL COMPARISON")
print("=" * 60)

comparison = []

# Make sure labels align with test predictions
y_true = y_test.to_numpy()

# Original RF probabilities
original_prob = test_results["refresh_probability"].to_numpy()

# Calibrated RF probabilities
cal_prob = np.asarray(calibrated_prob)

for threshold in [0.50, 0.55, 0.60]:

    # -------------------------
    # ORIGINAL RF
    # -------------------------
    orig_pred = (original_prob >= threshold).astype(int)

    orig_recommendations = orig_pred.sum()
    orig_actual_declines = y_true[orig_pred == 1].sum()

    orig_precision = (
        orig_actual_declines / orig_recommendations
        if orig_recommendations > 0 else 0
    )

    orig_recall = (
        orig_actual_declines / y_true.sum()
        if y_true.sum() > 0 else 0
    )

    # -------------------------
    # CALIBRATED RF
    # -------------------------
    cal_pred = (cal_prob >= threshold).astype(int)

    cal_recommendations = cal_pred.sum()
    cal_actual_declines = y_true[cal_pred == 1].sum()

    cal_precision = (
        cal_actual_declines / cal_recommendations
        if cal_recommendations > 0 else 0
    )

    cal_recall = (
        cal_actual_declines / y_true.sum()
        if y_true.sum() > 0 else 0
    )

    comparison.append({
        "threshold": threshold,

        "original_recommendations": orig_recommendations,
        "original_actual_declines": orig_actual_declines,
        "original_precision": round(orig_precision, 4),
        "original_recall": round(orig_recall, 4),

        "calibrated_recommendations": cal_recommendations,
        "calibrated_actual_declines": cal_actual_declines,
        "calibrated_precision": round(cal_precision, 4),
        "calibrated_recall": round(cal_recall, 4)
    })

comparison_df = pd.DataFrame(comparison)

print(comparison_df.to_string(index=False))

ORIGINAL RF VS CALIBRATED RF - FINAL COMPARISON
 threshold  original_recommendations  original_actual_declines  original_precision  original_recall  calibrated_recommendations  calibrated_actual_declines  calibrated_precision  calibrated_recall
      0.50                     11171                      4302              0.3851           0.5695                        8043                        4254                0.5289             0.5631
      0.55                      8803                      3658              0.4155           0.4842                        4672                        2599                0.5563             0.3441
      0.60                      6208                      3213              0.5176           0.4253                        1589                         925                0.5821             0.1225


In [86]:
print("client_eval columns:")
print(client_eval.columns.tolist())

print("\ntest_results_calibration columns:")
print(test_results_calibration.columns.tolist())

client_eval columns:
['impressions_90d', 'clicks_90d', 'avg_position_90d', 'impressions_last30', 'clicks_last30', 'ctr_90d', 'ctr_last30', 'content_age_days', 'days_since_update', 'search_volume', 'competition', 'backlinks', 'category_count', 'char_count', 'word_count', 'content_type', 'competition_level', 'main_intent', 'actual', 'prediction', 'probability']

test_results_calibration columns:
['impressions_90d', 'clicks_90d', 'avg_position_90d', 'impressions_last30', 'clicks_last30', 'ctr_90d', 'ctr_last30', 'content_age_days', 'days_since_update', 'search_volume', 'competition', 'backlinks', 'category_count', 'char_count', 'word_count', 'content_type', 'competition_level', 'main_intent', 'refresh_probability', 'client_hash_id', 'content_hash_id', 'rank', 'refresh_opportunity']


In [88]:
# Rebuild client evaluation table using test_results_calibration
# and the calibrated RF probabilities

client_eval = test_results_calibration[
    ["client_hash_id", "content_hash_id", "refresh_opportunity"]
].copy()

client_eval["actual"] = client_eval["refresh_opportunity"].astype(int)

client_eval["probability"] = calibrated_prob

# Threshold = 0.55
client_eval["prediction"] = (
    client_eval["probability"] >= 0.55
).astype(int)

print("CLIENT EVALUATION CREATED")
print("=" * 60)

print("Shape:", client_eval.shape)

print("\nColumns:")
print(client_eval.columns.tolist())

print("\nFirst 5 rows:")
print(client_eval.head())

print("\nPrediction distribution:")
print(client_eval["prediction"].value_counts())

print("\nActual distribution:")
print(client_eval["actual"].value_counts())

CLIENT EVALUATION CREATED
Shape: (18784, 6)

Columns:
['client_hash_id', 'content_hash_id', 'refresh_opportunity', 'actual', 'probability', 'prediction']

First 5 rows:
            client_hash_id           content_hash_id  refresh_opportunity  \
0  client_23a62021009f63c4  content_54b3b92802456c3f                    0   
1  client_23a62021009f63c4  content_bada26f0333b186a                    1   
2  client_23a62021009f63c4  content_c81aaca313704b4f                    0   
3  client_e5c2aa26a8598242  content_b88c5b2a685b5965                    0   
4  client_23a62021009f63c4  content_68d7bfebc3799aa5                    0   

   actual  probability  prediction  
0       0     0.339474           0  
1       1     0.484492           0  
2       0     0.445144           0  
3       0     0.528719           0  
4       0     0.476144           0  

Prediction distribution:
prediction
0    14112
1     4672
Name: count, dtype: int64

Actual distribution:
actual
0    11230
1     7554
Name: coun

In [89]:
print("=" * 60)
print("CALIBRATED RF CLIENT-LEVEL PERFORMANCE @ 0.55")
print("=" * 60)

client_summary = (
    client_eval
    .groupby("client_hash_id")
    .agg(
        total_pages=("actual", "size"),
        actual_declines=("actual", "sum"),
        recommendations=("prediction", "sum"),
        recommended_declines=("prediction", lambda x:
                              ((x == 1) &
                               (client_eval.loc[x.index, "actual"] == 1)).sum())
    )
)

# Precision among recommended pages
client_summary["precision_at_55"] = (
    client_summary["recommended_declines"] /
    client_summary["recommendations"]
).fillna(0)

# Client's natural decline rate
client_summary["baseline_decline_rate"] = (
    client_summary["actual_declines"] /
    client_summary["total_pages"]
)

# Lift over simply selecting pages randomly within that client
client_summary["lift_at_55"] = (
    client_summary["precision_at_55"] /
    client_summary["baseline_decline_rate"]
).replace([float("inf"), -float("inf")], 0)

print(client_summary.round(4))

print("\n" + "=" * 60)
print("AVERAGE CLIENT PERFORMANCE")
print("=" * 60)

print(
    "Mean Precision@0.55 per client:",
    round(client_summary["precision_at_55"].mean(), 4)
)

print(
    "Median Precision@0.55 per client:",
    round(client_summary["precision_at_55"].median(), 4)
)

print(
    "Mean Lift@0.55 per client:",
    round(client_summary["lift_at_55"].mean(), 4)
)

CALIBRATED RF CLIENT-LEVEL PERFORMANCE @ 0.55
                         total_pages  actual_declines  recommendations  \
client_hash_id                                                           
client_0b245132bb722950          357              140               85   
client_23a62021009f63c4         9634             3843             2473   
client_65de48885f4ef01b          744              303              206   
client_810019792c9b8efc          702              302              161   
client_86ebc2f12c01f586          915              370              222   
client_8ddc46da5414ffd8         2344             1002              592   
client_9958f0a7ae1df715         1576              596              332   
client_c7c2962f1c9c3089           26                6                1   
client_d211cb07b9059bab           29                8                9   
client_e5c2aa26a8598242         2457              984              591   

                         recommended_declines  precision_at_55  \

In [90]:
print("=" * 60)
print("CALIBRATED RF BUSINESS IMPACT @ 0.55")
print("=" * 60)

total_pages = len(client_eval)

total_declines = client_eval["actual"].sum()

recommendations = client_eval["prediction"].sum()

recommended_declines = (
    (client_eval["prediction"] == 1) &
    (client_eval["actual"] == 1)
).sum()

precision = recommended_declines / recommendations
recall = recommended_declines / total_declines

pages_not_recommended = total_pages - recommendations
declines_missed = total_declines - recommended_declines

print(f"Total pages:              {total_pages:,}")
print(f"Actual declines:          {total_declines:,}")
print(f"Refresh recommendations:  {recommendations:,}")
print(f"Recommended + declined:   {recommended_declines:,}")
print(f"Precision:                {precision:.4f}")
print(f"Recall:                   {recall:.4f}")
print(f"Pages not recommended:    {pages_not_recommended:,}")
print(f"Declines missed:           {declines_missed:,}")

print("\n" + "=" * 60)
print("REFRESH SAVINGS")
print("=" * 60)

print(
    f"Pages avoided from refresh: "
    f"{pages_not_recommended:,} "
    f"({pages_not_recommended / total_pages:.2%})"
)

print(
    f"Declining pages captured: "
    f"{recommended_declines:,} "
    f"({recall:.2%})"
)

CALIBRATED RF BUSINESS IMPACT @ 0.55
Total pages:              18,784
Actual declines:          7,554
Refresh recommendations:  4,672
Recommended + declined:   2,599
Precision:                0.5563
Recall:                   0.3441
Pages not recommended:    14,112
Declines missed:           4,955

REFRESH SAVINGS
Pages avoided from refresh: 14,112 (75.13%)
Declining pages captured: 2,599 (34.41%)


In [91]:
print("=" * 70)
print("FINAL THRESHOLD BUSINESS COMPARISON - CALIBRATED RF")
print("=" * 70)

thresholds = [0.50, 0.55, 0.60]

total_pages = len(client_eval)
total_declines = client_eval["actual"].sum()
baseline_rate = total_declines / total_pages

results = []

for threshold in thresholds:

    pred = (client_eval["probability"] >= threshold).astype(int)

    recommendations = pred.sum()

    captured = (
        (pred == 1) &
        (client_eval["actual"] == 1)
    ).sum()

    precision = captured / recommendations if recommendations > 0 else 0
    recall = captured / total_declines

    avoided = total_pages - recommendations

    lift = precision / baseline_rate

    results.append({
        "threshold": threshold,
        "recommendations": recommendations,
        "recommendation_pct": recommendations / total_pages,
        "actual_declines_captured": captured,
        "precision": precision,
        "recall": recall,
        "pages_avoided": avoided,
        "lift": lift
    })

final_threshold_comparison = pd.DataFrame(results)

print(final_threshold_comparison.round(4).to_string(index=False))

FINAL THRESHOLD BUSINESS COMPARISON - CALIBRATED RF
 threshold  recommendations  recommendation_pct  actual_declines_captured  precision  recall  pages_avoided   lift
      0.50             8043              0.4282                      4254     0.5289  0.5631          10741 1.3152
      0.55             4672              0.2487                      2599     0.5563  0.3441          14112 1.3833
      0.60             1589              0.0846                       925     0.5821  0.1225          17195 1.4475


In [92]:
print("=" * 70)
print("FINAL ORIGINAL RF VS CALIBRATED RF")
print("=" * 70)

comparison = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "Brier Score",
        "Log Loss",
        "Precision@100",
        "Precision@500",
        "Precision@1000",
        "Precision@2000",
        "Precision@5000"
    ],

    "Original_RF": [
        0.6522,
        0.2725,
        0.7431,
        0.6700,
        0.5800,
        0.5520,
        0.5615,
        0.5508
    ],

    "Calibrated_RF": [
        0.6568,
        0.2280,
        0.6474,
        0.5900,
        0.5560,
        0.5800,
        0.5775,
        0.5538
    ]
})

print(comparison.to_string(index=False))

FINAL ORIGINAL RF VS CALIBRATED RF
        Metric  Original_RF  Calibrated_RF
       ROC-AUC       0.6522         0.6568
   Brier Score       0.2725         0.2280
      Log Loss       0.7431         0.6474
 Precision@100       0.6700         0.5900
 Precision@500       0.5800         0.5560
Precision@1000       0.5520         0.5800
Precision@2000       0.5615         0.5775
Precision@5000       0.5508         0.5538


In [94]:
print("=" * 60)
print("CALIBRATED RF PIPELINE")
print("=" * 60)

print(calibrated_rf.estimator)
print("\nPipeline steps:")
print(calibrated_rf.estimator.named_steps)

CALIBRATED RF PIPELINE
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['impressions_90d',
                                                   'clicks_90d',
                                                   'avg_position_90d',
                                                   'impressions_last30',
                                                   'clicks_last30', 'ctr_90d',
                                                   'ctr_last30',
                                                   'content_age_days',
                                                   'days_since_update',
                                                   'search_volume',
                                                   'competition', '

In [95]:
import pandas as pd

print("=" * 60)
print("CALIBRATED RF FEATURE IMPORTANCE")
print("=" * 60)

# Get the fitted pipeline
pipeline = calibrated_rf.estimator

# Get the preprocessing and RF
preprocessor = pipeline.named_steps["preprocessor"]
rf_model = pipeline.named_steps["classifier"]

# Get transformed feature names
feature_names = preprocessor.get_feature_names_out()

# Get RF importance
importances = rf_model.feature_importances_

# Safety check
print("Number of transformed features:", len(feature_names))
print("Number of importance values:", len(importances))

# Create importance dataframe
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\nTop 30 features:")
print(feature_importance.head(30).to_string(index=False))

CALIBRATED RF FEATURE IMPORTANCE
Number of transformed features: 27
Number of importance values: 27

Top 30 features:
                                     feature  importance
                   numeric__avg_position_90d    0.145908
                   numeric__content_age_days    0.092655
                  numeric__days_since_update    0.081441
                         numeric__ctr_last30    0.073683
                         numeric__char_count    0.072758
                         numeric__clicks_90d    0.071124
                            numeric__ctr_90d    0.070886
                    numeric__impressions_90d    0.067430
                 numeric__impressions_last30    0.065774
                      numeric__clicks_last30    0.065672
                         numeric__word_count    0.062358
                      numeric__search_volume    0.036833
                        numeric__competition    0.031969
                     numeric__category_count    0.016893
                          n

In [96]:
print("=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

# 1. Check for future-related columns in model features
future_features = [
    col for col in X_train.columns
    if "future" in col.lower()
]

print("\nFuture-related model features:")
print(future_features)

# 2. Check exact model features
print("\nModel features:")
for col in X_train.columns:
    print(col)

# 3. Check target
print("\nTarget:")
print(y_train.name)

# 4. Check whether target exists in features
print("\nTarget accidentally in X?")
print(y_train.name in X_train.columns)

# 5. Check dataset columns containing future/target terms
print("\nDataset columns related to future/target:")
print([
    col for col in dataset_labeled.columns
    if "future" in col.lower()
    or "refresh" in col.lower()
])

LEAKAGE CHECK

Future-related model features:
[]

Model features:
impressions_90d
clicks_90d
avg_position_90d
impressions_last30
clicks_last30
ctr_90d
ctr_last30
content_age_days
days_since_update
search_volume
competition
backlinks
category_count
char_count
word_count
content_type
competition_level
main_intent

Target:
refresh_opportunity

Target accidentally in X?
False

Dataset columns related to future/target:
['future_impressions', 'future_clicks', 'future_avg_position', 'has_future_data']


In [97]:
print("=" * 70)
print("TEMPORAL DATA VALIDATION")
print("=" * 70)

print("Snapshot date range:")
print(dataset["snapshot_date"].min())
print(dataset["snapshot_date"].max())

print("\nSnapshot date distribution:")
print(
    dataset["snapshot_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

TEMPORAL DATA VALIDATION
Snapshot date range:
2026-05-31 00:00:00
2026-05-31 00:00:00

Snapshot date distribution:
snapshot_date
2026-05    163415
Freq: M, Name: count, dtype: int64


In [99]:
cutoff = dataset["snapshot_date"].quantile(0.80)

In [100]:
print("=" * 70)
print("FUTURE DATA VALIDATION")
print("=" * 70)

print("Total rows:", len(dataset))

print("\nHas future data:")
print(dataset["has_future_data"].value_counts(dropna=False))

print("\nFuture columns missing:")
print(
    dataset[
        ["future_impressions", "future_clicks", "future_avg_position"]
    ].isna().sum()
)

print("\nFuture impressions statistics:")
print(dataset["future_impressions"].describe())

print("\nFuture clicks statistics:")
print(dataset["future_clicks"].describe())

print("\nFuture position statistics:")
print(dataset["future_avg_position"].describe())

FUTURE DATA VALIDATION
Total rows: 163415

Has future data:
has_future_data
True     107602
False     55813
Name: count, dtype: int64

Future columns missing:
future_impressions     0
future_clicks          0
future_avg_position    0
dtype: int64

Future impressions statistics:
count    163415.000000
mean        423.668990
std        2727.466178
min           0.000000
25%           0.000000
50%           8.000000
75%         144.000000
max      235427.000000
Name: future_impressions, dtype: float64

Future clicks statistics:
count    163415.000000
mean          1.696644
std          11.563797
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        1102.000000
Name: future_clicks, dtype: float64

Future position statistics:
count    163415.000000
mean         15.766301
std          21.635649
min           0.000000
25%           0.000000
50%           7.108333
75%          22.027414
max         579.000000
Name: future_avg_position, dtype: fl

In [102]:
print("dataset columns:")
print(dataset.columns.tolist())

print("\ndataset_labeled columns:")
print(dataset_labeled.columns.tolist())

print("\ndataset_model columns:")
print(dataset_model.columns.tolist())

dataset columns:
['client_hash_id', 'content_hash_id', 'snapshot_date', 'impressions_90d', 'clicks_90d', 'avg_position_90d', 'impressions_last30', 'clicks_last30', 'content_created_date', 'content_updated_date', 'last_optimized_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'main_intent', 'backlinks', 'category_count', 'char_count', 'word_count', 'is_published', 'is_deleted', 'ctr_90d', 'ctr_last30', 'content_age_days', 'days_since_update', 'future_impressions', 'future_clicks', 'future_avg_position', 'has_future_data']

dataset_labeled columns:
['client_hash_id', 'content_hash_id', 'snapshot_date', 'impressions_90d', 'clicks_90d', 'avg_position_90d', 'impressions_last30', 'clicks_last30', 'content_created_date', 'content_updated_date', 'last_optimized_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'main_intent', 'backlinks', 'category_count', 'char_count', 'word_count', 'is_published', 'is_deleted', 'ctr_90d', 'ctr_last30', 'co

In [103]:
print("=" * 70)
print("FUTURE DATA + TARGET VALIDATION")
print("=" * 70)

df_check = dataset_model.copy()

print("\nShape:", df_check.shape)

print("\nTarget distribution:")
print(df_check["refresh_opportunity"].value_counts(dropna=False))

print("\nTarget rate:")
print(df_check["refresh_opportunity"].mean())

print("\nTarget vs future-data availability:")
print(
    pd.crosstab(
        df_check["has_future_data"],
        df_check["refresh_opportunity"],
        margins=True
    )
)

print("\nTarget rate by future-data availability:")
print(
    df_check.groupby("has_future_data")["refresh_opportunity"]
    .agg(["count", "mean"])
)

print("\nMissing target:")
print(df_check["refresh_opportunity"].isna().sum())

FUTURE DATA + TARGET VALIDATION

Shape: (71074, 31)

Target distribution:
refresh_opportunity
0    40476
1    30598
Name: count, dtype: int64

Target rate:
0.43050904690885555

Target vs future-data availability:
refresh_opportunity      0      1    All
has_future_data                         
True                 40476  30598  71074
All                  40476  30598  71074

Target rate by future-data availability:
                 count      mean
has_future_data                 
True             71074  0.430509

Missing target:
0


In [104]:
print("=" * 70)
print("REFRESH OPPORTUNITY TARGET DEFINITION")
print("=" * 70)

print("\nColumns related to target:")
print([
    c for c in dataset_labeled.columns
    if "impression" in c.lower()
    or "future" in c.lower()
    or "change" in c.lower()
])

print("\nImpression change statistics:")
print(
    dataset_labeled["impression_change_pct"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nImpression change by target:")
print(
    dataset_model.groupby("refresh_opportunity")[
        "future_impressions"
    ].agg(["count", "mean", "median", "min", "max"])
)

REFRESH OPPORTUNITY TARGET DEFINITION

Columns related to target:
['impressions_90d', 'impressions_last30', 'future_impressions', 'future_clicks', 'future_avg_position', 'has_future_data', 'impression_change_pct']

Impression change statistics:
count    105823.000000
mean          0.962703
std         593.410088
min         -99.987633
1%          -97.979798
5%          -93.243243
10%         -87.500000
25%         -70.923175
50%         -43.343109
75%           0.000000
90%          81.284609
95%         166.666667
99%         587.392857
max      112850.000000
Name: impression_change_pct, dtype: float64

Impression change by target:
                     count         mean  median   min       max
refresh_opportunity                                            
0                    40476  1511.367428   347.0  26.0  235427.0
1                    30598   244.278090    64.0   1.0   49076.0


In [106]:
["refresh_opportunity"]

['refresh_opportunity']

In [108]:
# Search columns in your dataframes
for name, obj in {
    "dataset": dataset,
    "dataset_labeled": dataset_labeled,
    "dataset_model": dataset_model
}.items():
    cols = [c for c in obj.columns if "refresh_opportunity" in c]
    print(name, ":", cols)

dataset : []
dataset_labeled : []
dataset_model : ['refresh_opportunity']


In [109]:
print("=" * 70)
print("REFRESH OPPORTUNITY TARGET VALIDATION")
print("=" * 70)

print("\nTarget distribution:")
print(dataset_model["refresh_opportunity"].value_counts())

print("\nTarget rate:")
print(dataset_model["refresh_opportunity"].mean())

print("\nTarget vs future data:")
print(
    pd.crosstab(
        dataset_model["has_future_data"],
        dataset_model["refresh_opportunity"],
        margins=True
    )
)

print("\nTarget rate by future-data availability:")
print(
    dataset_model.groupby("has_future_data")["refresh_opportunity"]
    .agg(["count", "mean"])
)

print("\nMissing target:")
print(dataset_model["refresh_opportunity"].isna().sum())

REFRESH OPPORTUNITY TARGET VALIDATION

Target distribution:
refresh_opportunity
0    40476
1    30598
Name: count, dtype: int64

Target rate:
0.43050904690885555

Target vs future data:
refresh_opportunity      0      1    All
has_future_data                         
True                 40476  30598  71074
All                  40476  30598  71074

Target rate by future-data availability:
                 count      mean
has_future_data                 
True             71074  0.430509

Missing target:
0


In [112]:
print("=" * 70)
print("TARGET FORMULA CHECK")
print("=" * 70)

check = dataset_model[
    [
        "impressions_90d",
        "impressions_last30",
        "future_impressions",
        "future_clicks",
        "future_avg_position",
        "has_future_data",
        "refresh_opportunity"
    ]
].copy()

# Calculate impression change temporarily
check["impression_change_pct"] = (
    (check["future_impressions"] - check["impressions_90d"])
    / check["impressions_90d"].replace(0, np.nan)
) * 100

print("\nTARGET = 1 EXAMPLES")
print(
    check[check["refresh_opportunity"] == 1]
    .head(15)
    .to_string(index=False)
)

print("\nTARGET = 0 EXAMPLES")
print(
    check[check["refresh_opportunity"] == 0]
    .head(15)
    .to_string(index=False)
)

TARGET FORMULA CHECK

TARGET = 1 EXAMPLES
 impressions_90d  impressions_last30  future_impressions  future_clicks  future_avg_position  has_future_data  refresh_opportunity  impression_change_pct
           541.0               541.0               207.0            0.0            16.506410             True                    1             -61.737523
           556.0               234.0               103.0            0.0            54.573923             True                    1             -81.474820
            82.0                63.0                16.0            0.0            19.658730             True                    1             -80.487805
           463.0               463.0                20.0            0.0            52.795455             True                    1             -95.680346
           409.0               409.0               148.0            0.0            36.291915             True                    1             -63.814181
           238.0               238

In [113]:
print("=" * 70)
print("TARGET THRESHOLD ANALYSIS")
print("=" * 70)

for threshold in [-90, -80, -70, -60, -50, -40, -30, -20, -10, 0]:
    predicted_target = (
        check["impression_change_pct"] <= threshold
    )

    actual_target = check["refresh_opportunity"] == 1

    agreement = (predicted_target == actual_target).mean()

    print(
        f"Threshold {threshold:>4}% -> "
        f"Agreement: {agreement:.4f}"
    )

TARGET THRESHOLD ANALYSIS
Threshold  -90% -> Agreement: 0.7800
Threshold  -80% -> Agreement: 0.7790
Threshold  -70% -> Agreement: 0.6954
Threshold  -60% -> Agreement: 0.6377
Threshold  -50% -> Agreement: 0.6005
Threshold  -40% -> Agreement: 0.5678
Threshold  -30% -> Agreement: 0.5449
Threshold  -20% -> Agreement: 0.5280
Threshold  -10% -> Agreement: 0.5148
Threshold    0% -> Agreement: 0.5041


In [115]:
dataset_model["refresh_opportunity"]

,refresh_opportunity
0,0
1,1
2,0
3,0
4,0
...,...
163334,0
163386,1
163389,0
163399,1


In [116]:
# Search variables that may contain the target-generation logic
print("Available variables containing 'refresh':")

[x for x in dir() if "refresh" in x.lower()]

Available variables containing 'refresh':


[]

In [117]:
# Check whether target-generation helper variables exist
for name in dir():
    if any(word in name.lower() for word in [
        "target", "label", "decline", "change", "future"
    ]):
        print(name)

actual_declines
actual_target
cal_actual_declines
dataset_labeled
decline
declined
declines_missed
future
future_columns
future_features
future_sql
orig_actual_declines
overall_decline_rate
predicted_target
recommended_declines
target_column
test_labels
total_declines


In [118]:
target_check = dataset_model[
    [
        "impressions_90d",
        "impressions_last30",
        "future_impressions",
        "future_clicks",
        "future_avg_position",
        "has_future_data",
        "refresh_opportunity"
    ]
].copy()

print(target_check.head(30).to_string())

    impressions_90d  impressions_last30  future_impressions  future_clicks  future_avg_position  has_future_data  refresh_opportunity
0            1148.0              1148.0               797.0            2.0            10.594463             True                    0
1             541.0               541.0               207.0            0.0            16.506410             True                    1
2             867.0               791.0              1252.0            5.0             7.442776             True                    0
3             627.0               627.0               515.0            4.0            19.613708             True                    0
4             560.0               524.0               816.0            1.0             7.958260             True                    0
5             367.0               228.0               142.0            1.0             7.331658             True                    0
6             360.0               360.0               220.0   

In [119]:
# Look for variables/functions containing target-related names
[
    x for x in globals()
    if any(k in x.lower() for k in [
        "refresh", "target", "label", "decline"
    ])
]

['dataset_labeled',
 'decline',
 'declined',
 'target_column',
 'overall_decline_rate',
 'actual_declines',
 'test_labels',
 'orig_actual_declines',
 'cal_actual_declines',
 'total_declines',
 'recommended_declines',
 'declines_missed',
 'predicted_target',
 'actual_target',
 'target_check']

In [120]:
print("=" * 70)
print("TARGET CONSISTENCY CHECK")
print("=" * 70)

df = dataset_model.copy()

df["impression_change_pct"] = (
    (df["future_impressions"] - df["impressions_90d"])
    / df["impressions_90d"].replace(0, np.nan)
) * 100

# Examine target rate at different decline thresholds
for threshold in [-95, -90, -85, -80, -75, -70, -65, -60]:
    rule = df["impression_change_pct"] <= threshold

    print(
        f"Threshold {threshold:>4}% | "
        f"rule positives: {rule.sum():6d} | "
        f"target positives: {df['refresh_opportunity'].sum():6d} | "
        f"agreement: {(rule == df['refresh_opportunity']).mean():.4f}"
    )

TARGET CONSISTENCY CHECK
Threshold  -95% | rule positives:   9659 | target positives:  30598 | agreement: 0.6913
Threshold  -90% | rule positives:  20814 | target positives:  30598 | agreement: 0.7800
Threshold  -85% | rule positives:  31080 | target positives:  30598 | agreement: 0.8032
Threshold  -80% | rule positives:  38867 | target positives:  30598 | agreement: 0.7790
Threshold  -75% | rule positives:  44775 | target positives:  30598 | agreement: 0.7366
Threshold  -70% | rule positives:  49340 | target positives:  30598 | agreement: 0.6954
Threshold  -65% | rule positives:  52807 | target positives:  30598 | agreement: 0.6640
Threshold  -60% | rule positives:  55379 | target positives:  30598 | agreement: 0.6377


In [121]:
print("=" * 70)
print("TARGET vs -85% RULE: DISAGREEMENTS")
print("=" * 70)

threshold = -85

rule = df["impression_change_pct"] <= threshold

disagreements = df.loc[
    rule != df["refresh_opportunity"],
    [
        "impressions_90d",
        "impressions_last30",
        "future_impressions",
        "future_clicks",
        "future_avg_position",
        "impression_change_pct",
        "refresh_opportunity"
    ]
].copy()

print("Total disagreements:", len(disagreements))
print(
    "Disagreement rate:",
    len(disagreements) / len(df)
)

print("\nRule = 1 but Target = 0:")
print(
    disagreements[
        (disagreements["impression_change_pct"] <= -85) &
        (disagreements["refresh_opportunity"] == 0)
    ].head(30).to_string(index=False)
)

print("\nRule = 0 but Target = 1:")
print(
    disagreements[
        (disagreements["impression_change_pct"] > -85) &
        (disagreements["refresh_opportunity"] == 1)
    ].head(30).to_string(index=False)
)

TARGET vs -85% RULE: DISAGREEMENTS
Total disagreements: 13984
Disagreement rate: 0.19675268030503418

Rule = 1 but Target = 0:
 impressions_90d  impressions_last30  future_impressions  future_clicks  future_avg_position  impression_change_pct  refresh_opportunity
          1072.0               310.0               156.0            0.0             8.189739             -85.447761                    0
           391.0                98.0                53.0            0.0             9.416667             -86.445013                    0
           448.0                86.0                57.0            0.0            19.794444             -87.276786                    0
          1222.0               314.0               177.0            1.0            20.173179             -85.515548                    0
           456.0                94.0                51.0            0.0            20.059028             -88.815789                    0
          3660.0               872.0               

In [122]:
print("=" * 70)
print("TARGET RELATIONSHIP WITH FUTURE VARIABLES")
print("=" * 70)

cols = [
    "impression_change_pct",
    "future_impressions",
    "future_clicks",
    "future_avg_position"
]

for col in cols:
    print(f"\n--- {col} ---")

    print(
        df.groupby("refresh_opportunity")[col]
          .agg(["count", "mean", "median", "min", "max"])
    )

    print("\nQuantiles:")
    print(
        df.groupby("refresh_opportunity")[col]
          .quantile([0.10, 0.25, 0.50, 0.75, 0.90])
          .unstack()
    )

TARGET RELATIONSHIP WITH FUTURE VARIABLES

--- impression_change_pct ---
                     count       mean     median        min          max
refresh_opportunity                                                     
0                    40476 -41.252034 -69.936709 -99.950191  3552.380952
1                    30598 -89.442560 -91.624191 -99.999308   -50.000000

Quantiles:
                          0.10       0.25       0.50       0.75       0.90
refresh_opportunity                                                       
0                   -88.550702 -82.012531 -69.936709 -41.666667  21.309943
1                   -98.171344 -95.735010 -91.624191 -86.004529 -78.055709

--- future_impressions ---
                     count         mean  median   min       max
refresh_opportunity                                            
0                    40476  1511.367428   347.0  26.0  235427.0
1                    30598   244.278090    64.0   1.0   49076.0

Quantiles:
                     0.10  

In [123]:
print("=" * 70)
print("TARGET RULE SEARCH")
print("=" * 70)

# Test combinations of decline thresholds and future-impression thresholds

decline_thresholds = [-50, -60, -70, -75, -80, -85, -90, -95]
future_imp_thresholds = [10, 20, 50, 100, 200, 500]

results = []

for d in decline_thresholds:
    for f in future_imp_thresholds:

        rule = (
            (df["impression_change_pct"] <= d) &
            (df["future_impressions"] <= f)
        )

        agreement = (rule == df["refresh_opportunity"]).mean()

        results.append({
            "decline_threshold": d,
            "future_impressions_threshold": f,
            "rule_positives": rule.sum(),
            "agreement": agreement
        })

rule_results = (
    pd.DataFrame(results)
    .sort_values("agreement", ascending=False)
)

print(rule_results.head(20).to_string(index=False))

TARGET RULE SEARCH
 decline_threshold  future_impressions_threshold  rule_positives  agreement
               -85                           500           27431   0.796001
               -80                           500           33393   0.785871
               -80                           200           25953   0.778189
               -85                           200           22050   0.777950
               -90                           500           19021   0.768931
               -75                           200           28634   0.765906
               -80                           100           19681   0.763641
               -75                           500           37517   0.762177
               -75                           100           21205   0.760208
               -85                           100           17285   0.757140
               -70                           100           22168   0.756507
               -70                           200           30415   0.

In [124]:
print("=" * 70)
print("FULL DATASET TIMELINE AUDIT")
print("=" * 70)

print("\nSnapshot date:")
print(dataset_model["snapshot_date"].min())
print(dataset_model["snapshot_date"].max())

print("\nUnique snapshot dates:")
print(dataset_model["snapshot_date"].nunique())

print("\nSnapshot distribution:")
print(
    dataset_model["snapshot_date"]
    .value_counts()
    .sort_index()
)

print("\nContent created date:")
print(dataset_model["content_created_date"].min())
print(dataset_model["content_created_date"].max())

print("\nContent updated date:")
print(dataset_model["content_updated_date"].min())
print(dataset_model["content_updated_date"].max())

print("\nLast optimized date:")
print(dataset_model["last_optimized_date"].min())
print(dataset_model["last_optimized_date"].max())

FULL DATASET TIMELINE AUDIT

Snapshot date:
2026-05-31 00:00:00
2026-05-31 00:00:00

Unique snapshot dates:
1

Snapshot distribution:
snapshot_date
2026-05-31    71074
Name: count, dtype: int64

Content created date:
2024-11-22 00:00:00
2026-05-20 00:00:00

Content updated date:
2025-07-10 00:00:00
2026-05-29 00:00:00

Last optimized date:
2026-04-24 00:00:00
2026-05-29 00:00:00


In [125]:
print("=" * 70)
print("TRAIN / TEST CONTENT OVERLAP CHECK")
print("=" * 70)

train_content = set(X_train.index)
test_content = set(X_test.index)

print("Train rows:", len(train_content))
print("Test rows:", len(test_content))

TRAIN / TEST CONTENT OVERLAP CHECK
Train rows: 52290
Test rows: 18784


In [126]:
print("=" * 70)
print("TRAIN / TEST CONTENT HASH OVERLAP")
print("=" * 70)

# Get IDs from the same rows used for train/test
train_content_ids = set(
    dataset_model.loc[X_train.index, "content_hash_id"]
)

test_content_ids = set(
    dataset_model.loc[X_test.index, "content_hash_id"]
)

overlap = train_content_ids.intersection(test_content_ids)

print("Unique train contents:", len(train_content_ids))
print("Unique test contents:", len(test_content_ids))
print("Overlapping contents:", len(overlap))

if len(overlap) == 0:
    print("\n✅ NO CONTENT OVERLAP")
    print("Train and test contain completely different content IDs.")
else:
    print("\n⚠️ CONTENT OVERLAP DETECTED")
    print("This can cause optimistic test performance.")
    print("Number of overlapping content IDs:", len(overlap))

TRAIN / TEST CONTENT HASH OVERLAP
Unique train contents: 52290
Unique test contents: 18784
Overlapping contents: 0

✅ NO CONTENT OVERLAP
Train and test contain completely different content IDs.


In [131]:
subgroup_eval = client_eval.copy()

subgroup_eval = subgroup_eval.merge(
    test_results_calibration[
        [
            "content_hash_id",
            "content_type",
            "competition_level",
            "main_intent"
        ]
    ],
    on="content_hash_id",
    how="left"
)

# Prediction at threshold 0.55
subgroup_eval["prediction"] = (
    subgroup_eval["probability"] >= 0.55
).astype(int)



def evaluate_subgroup(df, column):

    rows = []

    for group_name, group in df.groupby(column, dropna=False):

        pages = len(group)
        actual_declines = group["actual"].sum()
        recommendations = group["prediction"].sum()

        recommended_declines = group.loc[
            group["prediction"] == 1,
            "actual"
        ].sum()

        precision = (
            recommended_declines / recommendations
            if recommendations > 0
            else 0
        )

        recall = (
            recommended_declines / actual_declines
            if actual_declines > 0
            else 0
        )

        baseline_rate = (
            actual_declines / pages
            if pages > 0
            else 0
        )

        lift = (
            precision / baseline_rate
            if baseline_rate > 0
            else 0
        )

        rows.append({
            column: group_name,
            "pages": pages,
            "actual_declines": actual_declines,
            "recommendations": recommendations,
            "recommended_declines": recommended_declines,
            "precision": precision,
            "recall": recall,
            "baseline_rate": baseline_rate,
            "lift": lift
        })

    return pd.DataFrame(rows)




for column in [
    "content_type",
    "competition_level",
    "main_intent"
]:

    print("\n" + "=" * 70)
    print(f"BY {column.upper()}")
    print("=" * 70)

    result = evaluate_subgroup(
        subgroup_eval,
        column
    )

    print(result.to_string(index=False))


BY CONTENT_TYPE
   content_type  pages  actual_declines  recommendations  recommended_declines  precision   recall  baseline_rate     lift
 feedly article    465              204              141                    85   0.602837 0.416667       0.438710 1.374113
keyword article  18319             7350             4531                  2514   0.554844 0.342041       0.401223 1.382884

BY COMPETITION_LEVEL
competition_level  pages  actual_declines  recommendations  recommended_declines  precision   recall  baseline_rate     lift
             HIGH    694              305              193                   105   0.544041 0.344262       0.439481 1.237917
              LOW  16749             6693             4157                  2308   0.555208 0.344838       0.399606 1.389389
           MEDIUM    588              239              121                    68   0.561983 0.284519       0.406463 1.382620
              NaN    753              317              201                   118   0.587065 

In [132]:
print("=" * 70)
print("FINAL OPERATING THRESHOLD DECISION")
print("=" * 70)

thresholds = [0.50, 0.55, 0.60]

for t in thresholds:

    pred = (
        test_results_calibration["refresh_probability"] >= t
    ).astype(int)

    actual = client_eval["actual"].values

    recommendations = pred.sum()
    actual_declines = actual.sum()
    captured = ((pred == 1) & (actual == 1)).sum()

    precision = (
        captured / recommendations
        if recommendations > 0 else 0
    )

    recall = (
        captured / actual_declines
        if actual_declines > 0 else 0
    )

    recommendation_pct = (
        recommendations / len(pred)
    )

    print(f"\nThreshold: {t:.2f}")
    print(f"Recommendations:       {recommendations:,}")
    print(f"Recommendation rate:   {recommendation_pct:.2%}")
    print(f"Declines captured:     {captured:,}")
    print(f"Precision:             {precision:.4f}")
    print(f"Recall:                {recall:.4f}")

FINAL OPERATING THRESHOLD DECISION

Threshold: 0.50
Recommendations:       11,171
Recommendation rate:   59.47%
Declines captured:     4,302
Precision:             0.3851
Recall:                0.5695

Threshold: 0.55
Recommendations:       8,803
Recommendation rate:   46.86%
Declines captured:     3,658
Precision:             0.4155
Recall:                0.4842

Threshold: 0.60
Recommendations:       6,208
Recommendation rate:   33.05%
Declines captured:     3,213
Precision:             0.5176
Recall:                0.4253


In [133]:
pred = (
    test_results_calibration["refresh_probability"] >= t
).astype(int)

In [134]:
print("=" * 70)
print("PROBABILITY SOURCE CHECK")
print("=" * 70)

print("test_results_calibration columns:")
print(test_results_calibration.columns.tolist())

print("\nProbability summary:")
print(test_results_calibration["refresh_probability"].describe())

print("\nFirst 10 probabilities:")
print(
    test_results_calibration["refresh_probability"]
    .head(10)
    .to_list()
)

print("\nCalibrated RF object:")
print(calibrated_rf)

PROBABILITY SOURCE CHECK
test_results_calibration columns:
['impressions_90d', 'clicks_90d', 'avg_position_90d', 'impressions_last30', 'clicks_last30', 'ctr_90d', 'ctr_last30', 'content_age_days', 'days_since_update', 'search_volume', 'competition', 'backlinks', 'category_count', 'char_count', 'word_count', 'content_type', 'competition_level', 'main_intent', 'refresh_probability', 'client_hash_id', 'content_hash_id', 'rank', 'refresh_opportunity']

Probability summary:
count    18784.000000
mean         0.515991
std          0.150469
min          0.071154
25%          0.420654
50%          0.538208
75%          0.628933
max          0.871882
Name: refresh_probability, dtype: float64

First 10 probabilities:
[0.8718817622433551, 0.8635957370945007, 0.8633099784985202, 0.8588590756262661, 0.85740258932593, 0.8570431640424581, 0.8550157627821502, 0.8488674971941688, 0.8476808996553166, 0.8474248909760721]

Calibrated RF object:
CalibratedClassifierCV(cv=3,
                       estimator

In [135]:
print("=" * 70)
print("FINAL CALIBRATED RF THRESHOLD COMPARISON")
print("=" * 70)

y_actual = test_results_calibration["refresh_opportunity"].values
y_prob = test_results_calibration["refresh_probability"].values

for threshold in [0.50, 0.55, 0.60]:

    y_pred = (y_prob >= threshold).astype(int)

    recommendations = y_pred.sum()
    actual_declines = y_actual.sum()

    captured = ((y_pred == 1) & (y_actual == 1)).sum()

    precision = captured / recommendations if recommendations else 0
    recall = captured / actual_declines if actual_declines else 0

    pages_avoided = len(y_actual) - recommendations

    print(f"\nThreshold: {threshold:.2f}")
    print(f"Recommendations:       {recommendations:,}")
    print(f"Recommendation rate:   {recommendations/len(y_actual):.2%}")
    print(f"Actual declines:       {actual_declines:,}")
    print(f"Declines captured:     {captured:,}")
    print(f"Precision:             {precision:.4f}")
    print(f"Recall:                {recall:.4f}")
    print(f"Pages avoided:         {pages_avoided:,}")

FINAL CALIBRATED RF THRESHOLD COMPARISON

Threshold: 0.50
Recommendations:       11,171
Recommendation rate:   59.47%
Actual declines:       7,554
Declines captured:     4,302
Precision:             0.3851
Recall:                0.5695
Pages avoided:         7,613

Threshold: 0.55
Recommendations:       8,803
Recommendation rate:   46.86%
Actual declines:       7,554
Declines captured:     3,658
Precision:             0.4155
Recall:                0.4842
Pages avoided:         9,981

Threshold: 0.60
Recommendations:       6,208
Recommendation rate:   33.05%
Actual declines:       7,554
Declines captured:     3,213
Precision:             0.5176
Recall:                0.4253
Pages avoided:         12,576


In [136]:


import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL CALIBRATED RANDOM FOREST MODEL")
print("=" * 70)

print("\nMODEL:")
print(calibrated_rf)

print("\nDATASET:")
print(f"Test rows: {len(test_results_calibration):,}")
print(f"Actual declines: {y_actual.sum():,}")
print(f"Actual decline rate: {y_actual.mean():.2%}")

print("\nFINAL OPERATING THRESHOLD")
threshold = 0.60

final_prediction = (y_prob >= threshold).astype(int)

recommendations = final_prediction.sum()
captured = ((final_prediction == 1) & (y_actual == 1)).sum()
actual_declines = y_actual.sum()

precision = captured / recommendations
recall = captured / actual_declines

print(f"Threshold:             {threshold}")
print(f"Recommendations:       {recommendations:,}")
print(f"Recommendation rate:   {recommendations / len(y_actual):.2%}")
print(f"Declines captured:     {captured:,}")
print(f"Precision:             {precision:.4f}")
print(f"Recall:                {recall:.4f}")
print(f"Pages avoided:         {len(y_actual)-recommendations:,}")
print(f"Pages avoided rate:    {(len(y_actual)-recommendations)/len(y_actual):.2%}")

print("\nPREDICTION DISTRIBUTION:")
print(pd.Series(final_prediction).value_counts())

print("\nACTUAL DISTRIBUTION:")
print(pd.Series(y_actual).value_counts())

FINAL CALIBRATED RANDOM FOREST MODEL

MODEL:
CalibratedClassifierCV(cv=3,
                       estimator=Pipeline(steps=[('preprocessor',
                                                  ColumnTransformer(transformers=[('numeric',
                                                                                   Pipeline(steps=[('imputer',
                                                                                                    SimpleImputer(strategy='median'))]),
                                                                                   ['impressions_90d',
                                                                                    'clicks_90d',
                                                                                    'avg_position_90d',
                                                                                    'impressions_last30',
                                                                                    'clicks_last30',
      

In [137]:


final_recommendations = test_results_calibration.copy()

final_recommendations["refresh_probability"] = y_prob
final_recommendations["refresh_recommendation"] = (
    y_prob >= 0.60
).astype(int)

final_recommendations = final_recommendations.sort_values(
    "refresh_probability",
    ascending=False
)

print("=" * 70)
print("FINAL REFRESH RECOMMENDATIONS")
print("=" * 70)

print(
    final_recommendations[
        [
            "client_hash_id",
            "content_hash_id",
            "refresh_probability",
            "refresh_recommendation",
            "refresh_opportunity"
        ]
    ].head(20)
)

FINAL REFRESH RECOMMENDATIONS
             client_hash_id           content_hash_id  refresh_probability  \
0   client_23a62021009f63c4  content_54b3b92802456c3f             0.871882   
1   client_23a62021009f63c4  content_bada26f0333b186a             0.863596   
2   client_23a62021009f63c4  content_c81aaca313704b4f             0.863310   
3   client_e5c2aa26a8598242  content_b88c5b2a685b5965             0.858859   
4   client_23a62021009f63c4  content_68d7bfebc3799aa5             0.857403   
5   client_23a62021009f63c4  content_c7b938c9146b82f8             0.857043   
6   client_e5c2aa26a8598242  content_c56835932e398b5a             0.855016   
7   client_23a62021009f63c4  content_9afac8ac394aca4f             0.848867   
8   client_23a62021009f63c4  content_4b629b107ed54970             0.847681   
9   client_23a62021009f63c4  content_d08fdd23a06e49d0             0.847425   
10  client_23a62021009f63c4  content_2fad4f04929194cb             0.845530   
11  client_23a62021009f63c4  conte

In [138]:


recommendation_columns = [
    "client_hash_id",
    "content_hash_id",
    "refresh_probability",
    "refresh_recommendation",
    "refresh_opportunity"
]

final_recommendations[
    recommendation_columns
].to_csv(
    "final_refresh_recommendations.csv",
    index=False
)

print("=" * 70)
print("FINAL RECOMMENDATION EXPORT")
print("=" * 70)

print("Saved: final_refresh_recommendations.csv")
print(
    f"Rows exported: "
    f"{len(final_recommendations):,}"
)

print(
    f"Recommended for refresh: "
    f"{final_recommendations['refresh_recommendation'].sum():,}"
)

print(
    f"Not recommended: "
    f"{(final_recommendations['refresh_recommendation'] == 0).sum():,}"
)

FINAL RECOMMENDATION EXPORT
Saved: final_refresh_recommendations.csv
Rows exported: 18,784
Recommended for refresh: 6,208
Not recommended: 12,576


In [141]:


print("=" * 70)
print("FINAL SANITY CHECK")
print("=" * 70)

checks = {}

# ------------------------------------------------------------
# 1. MODEL FEATURES
# ------------------------------------------------------------

model_features_check = [
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d",
    "impressions_last30",
    "clicks_last30",
    "ctr_90d",
    "ctr_last30",
    "content_age_days",
    "days_since_update",
    "search_volume",
    "competition",
    "backlinks",
    "category_count",
    "char_count",
    "word_count",
    "content_type",
    "competition_level",
    "main_intent"
]

future_features = [
    "future_impressions",
    "future_clicks",
    "future_avg_position",
    "impression_change_pct",
    "has_future_data"
]

checks["No target in model features"] = (
    "refresh_opportunity" not in model_features_check
)

checks["No future features in model"] = not any(
    f in model_features_check
    for f in future_features
)

# ------------------------------------------------------------
# 2. TEST DATA
# ------------------------------------------------------------

checks["Test rows = 18,784"] = (
    len(test_results_calibration) == 18784
)

checks["Actual declines = 7,554"] = (
    int(test_results_calibration["refresh_opportunity"].sum())
    == 7554
)

# ------------------------------------------------------------
# 3. PREDICTIONS
# ------------------------------------------------------------

checks["Prediction count = test rows"] = (
    len(y_prob) == 18784
)

checks["No missing probabilities"] = (
    not pd.Series(y_prob).isna().any()
)

checks["No missing actual labels"] = (
    not pd.Series(
        test_results_calibration["refresh_opportunity"]
    ).isna().any()
)

# ------------------------------------------------------------
# 4. THRESHOLD = 0.60
# ------------------------------------------------------------

predicted_final = (y_prob >= 0.60).astype(int)

checks["0.60 recommendations = 6,208"] = (
    predicted_final.sum() == 6208
)

checks["0.60 not recommended = 12,576"] = (
    (predicted_final == 0).sum() == 12576
)

# ------------------------------------------------------------
# 5. EXPORTED FILE
# ------------------------------------------------------------

checks["Export rows = 18,784"] = (
    len(final_recommendations) == 18784
)

checks["Export recommendations = 6,208"] = (
    final_recommendations[
        "refresh_recommendation"
    ].sum() == 6208
)

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

for name, result in checks.items():
    print(f"{'✅' if result else '❌'} {name}")

print("\n" + "=" * 70)

if all(checks.values()):
    print("✅ ALL FINAL SANITY CHECKS PASSED")
    print("✅ MODEL IS READY FOR FINAL REPORTING")
else:
    print("❌ SOME CHECKS FAILED")
    print("DO NOT FINALIZE YET.")

FINAL SANITY CHECK
✅ No target in model features
✅ No future features in model
✅ Test rows = 18,784
✅ Actual declines = 7,554
✅ Prediction count = test rows
✅ No missing probabilities
✅ No missing actual labels
✅ 0.60 recommendations = 6,208
✅ 0.60 not recommended = 12,576
✅ Export rows = 18,784
✅ Export recommendations = 6,208

✅ ALL FINAL SANITY CHECKS PASSED
✅ MODEL IS READY FOR FINAL REPORTING


## 1. Question

*The research question and the decision it supports.*

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
